# Loop Transformer experiments

Four-pass Ouro-1.4B, a frozen Qwen2.5-1.5B comparison, and a controlled small-model study. Checkpoint weights and dependencies live in Kaggle temporary storage. The results directory contains the measurements and validation records.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

work = Path("/kaggle/working")
temp = Path("/kaggle/temp")
src = temp / "loop-source"
lib = temp / "loop-py"
out = work / "results"
for directory in (src, lib, out):
    directory.mkdir(parents=True, exist_ok=True)
assert torch.cuda.device_count(), "Select a GPU accelerator in Kaggle Session options"
free = {str(p): round(shutil.disk_usage(p).free / 2**30, 2) for p in (temp, work)}
print("Free GiB", free, flush=True)
assert free[str(temp)] > 8 and free[str(work)] > 1, "Insufficient free storage"
env = dict(os.environ, HF_HOME=str(temp / "loop-hf"),
           HF_HUB_CACHE=str(temp / "loop-hf" / "hub"), HF_HUB_DISABLE_XET="1",
           PYTHONPATH=str(lib) + os.pathsep + os.environ.get("PYTHONPATH", ""),
           OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
           MPLBACKEND="Agg")
deps = ["transformers==4.55.0", "tokenizers==0.21.4", "huggingface-hub==0.36.0",
        "accelerate==1.10.1"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                "--no-deps", "--target", str(lib), *deps], check=True)
print("GPUs", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], flush=True)

In [ ]:
%%writefile /kaggle/temp/loop-source/model_engine.py
"""Frozen, no-cache forward passes for the four-stage checkpoint experiments.

Sites are (one-based stage, one-based block offset). Captures and edits apply
after the physical decoder block and before any stage-end RMSNorm. Ouro feeds
its normalized state into the next pass. Qwen feeds its residual stream into
the next layer and normalizes only for an output-head readout.
"""
from contextlib import nullcontext
from dataclasses import dataclass

import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM, AutoTokenizer


CHECKPOINTS = {
    "ouro": ("ByteDance/Ouro-1.4B", "574fa66cb8bf5abdc979642d01cf2b79b16bfab1"),
    "qwen": ("Qwen/Qwen2.5-1.5B", "8faed761d45a263340a0528343f099c05c9a4323"),
}


@dataclass
class Run:
    logits: torch.Tensor
    states: dict
    engine: object

    def stage_logits(self, stage, positions=None):
        """Project a captured stage boundary through the frozen output head."""
        hidden = self.states[(stage, self.engine.width)]
        if positions is not None:
            hidden = hidden[:, positions, :]
        return self.engine.readout(hidden)


class Engine:
    def __init__(self, name, model, tokenizer):
        self.name = name
        self.model = model.eval().requires_grad_(False)
        self.tokenizer = tokenizer
        self.device = model.get_input_embeddings().weight.device
        self.dtype = model.get_input_embeddings().weight.dtype
        layers = model.model.layers
        if name == "ouro":
            if model.model.total_ut_steps != 4:
                raise ValueError("Expected four native Ouro passes")
            self.width = len(layers)
        elif name == "qwen" and len(layers) % 4 == 0:
            self.width = len(layers) // 4
        else:
            raise ValueError("Expected Ouro or a Qwen stack divisible into four stages")
        self.sdpa = model.config._attn_implementation == "sdpa"
        if model.config._attn_implementation not in ("eager", "sdpa"):
            raise ValueError("This adapter requires eager or math SDPA attention")
        if any(layer.attention_type != "full_attention" for layer in layers):
            raise ValueError("This experiment assumes full causal attention throughout")

    @classmethod
    def load(cls, name):
        if not torch.cuda.is_available():
            raise RuntimeError("Select a Kaggle GPU accelerator before loading weights")
        repo, revision = CHECKPOINTS[name]
        tokenizer = AutoTokenizer.from_pretrained(repo, revision=revision, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            repo,
            revision=revision,
            code_revision=revision,
            trust_remote_code=(name == "ouro"),
            use_safetensors=True,
            torch_dtype=torch.float16,
            attn_implementation="eager" if name == "ouro" else "sdpa",
            device_map={"": "cuda:0"},
        )
        return cls(name, model, tokenizer)

    def ids(self, text):
        return self.tokenizer.encode(text, add_special_tokens=False)

    def readout(self, hidden):
        return self.model.lm_head(self.model.model.norm(hidden))

    def attention_context(self):
        # T4 math SDPA accumulates FP16 QK, softmax and AV in FP32 by default.
        # Force the same kernel for native causal and experimental 4-D masks.
        return sdpa_kernel(SDPBackend.MATH) if self.sdpa else nullcontext()

    def run(self, ids, *, mask_fn=None, edit_fn=None, capture=(), grad=False,
            position_offset=0, logits_positions=None):
        """Run four full stages on one sequence.

        mask_fn receives an independently writable additive causal mask
        [1,1,query,key]. edit_fn may return a replacement state or None.
        For sparse output, pass a slice/list as logits_positions.
        """
        ids = torch.as_tensor(ids, device=self.device, dtype=torch.long).reshape(1, -1)
        length = ids.shape[1]
        if length == 0 or position_offset < 0:
            raise ValueError("Expected a nonempty sequence and nonnegative RoPE offset")
        sites = set(capture)
        if any(not (1 <= stage <= 4 and 1 <= offset <= self.width)
               for stage, offset in sites):
            raise ValueError(f"Capture site outside this checkpoint: {sites}")
        backbone = self.model.model
        cache_position = torch.arange(length, device=self.device)
        positions = (cache_position + position_offset).unsqueeze(0)
        states = {}
        with torch.set_grad_enabled(grad), self.attention_context():
            hidden = backbone.embed_tokens(ids)
            # Frozen parameters need a differentiable input to expose gradients.
            if grad:
                hidden.requires_grad_(True)
            rotary = backbone.rotary_emb(hidden, positions)
            blocked = float("-inf") if self.sdpa else torch.finfo(self.dtype).min
            causal = torch.full((length, length), blocked,
                                dtype=self.dtype, device=self.device)
            causal = causal.triu(1)[None, None, :, :]
            for stage in range(1, 5):
                start = 0 if self.name == "ouro" else (stage - 1) * self.width
                extra = {"current_ut": stage - 1} if self.name == "ouro" else {}
                for index in range(self.width):
                    site = (stage, index + 1)
                    mask = causal if mask_fn is None else mask_fn(*site, causal.clone())
                    if self.sdpa and mask_fn is not None:
                        mask = mask.masked_fill(mask == torch.finfo(mask.dtype).min, float("-inf"))
                    hidden = backbone.layers[start + index](
                        hidden,
                        attention_mask=mask,
                        position_ids=positions,
                        cache_position=cache_position,
                        position_embeddings=rotary,
                        use_cache=False,
                        **extra,
                    )
                    if edit_fn is not None:
                        replacement = edit_fn(*site, hidden)
                        if replacement is not None:
                            hidden = replacement
                    if site in sites:
                        states[site] = hidden
                if self.name == "ouro" or stage == 4:
                    hidden = backbone.norm(hidden)
            if logits_positions is not None:
                hidden = hidden[:, logits_positions, :]
            logits = self.model.lm_head(hidden)
        return Run(logits, states, self)

    @torch.no_grad()
    def validate(self, ids=None):
        """Fail before measurement if native, manual, or sham executions diverge."""
        if ids is None:
            ids = self.ids("The blue box contains a pear. It contains a")
        if self.sdpa and torch.backends.cuda.fp16_bf16_reduction_math_sdp_allowed():
            torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        x = torch.tensor(ids, device=self.device).reshape(1, -1)
        boundaries = [(stage, self.width) for stage in range(1, 5)]

        def finite(tensor, label):
            if not torch.isfinite(tensor).all():
                bad = (~torch.isfinite(tensor)).nonzero()[0].tolist()
                raise FloatingPointError(f"{self.name} {self.dtype} {label} nonfinite at {bad}")

        def check_block(stage, offset, hidden):
            finite(hidden, f"manual stage {stage}, block {offset}")

        manual = self.run(ids, capture=boundaries, edit_fn=check_block)
        finite(manual.logits, "manual output head")
        native_states = {}
        handles = []
        for index, layer in enumerate(self.model.model.layers):
            def record(module, args, kwargs, value, physical=index):
                if self.name == "ouro":
                    site = (kwargs["current_ut"] + 1, physical + 1)
                else:
                    group, within = divmod(physical, self.width)
                    site = (group + 1, within + 1)
                finite(value, f"native stage {site[0]}, block {site[1]}")
                if site in boundaries:
                    native_states[site] = value
            handles.append(layer.register_forward_hook(record, with_kwargs=True))
        try:
            extra = {"exit_at_step": 3} if self.name == "ouro" else {}
            with self.attention_context():
                native = self.model(input_ids=x, use_cache=False, **extra).logits
        finally:
            for handle in handles:
                handle.remove()
        sham = self.run(ids, mask_fn=lambda stage, offset, mask: mask,
                        edit_fn=lambda stage, offset, hidden: hidden).logits

        def max_error(actual, expected):
            finite(actual, "comparison output")
            finite(expected, "native output")
            error = (actual.float() - expected.float()).abs().max().item()
            torch.testing.assert_close(actual, expected, atol=0.002, rtol=0.001)
            return error

        return {
            "checkpoint": CHECKPOINTS[self.name],
            "blocks_per_stage": self.width,
            "executed_depth": self.width * 4,
            "final_native_max_abs_error": max_error(manual.logits, native),
            "stage_native_max_abs_errors": [
                max_error(manual.states[k], native_states[k]) for k in boundaries
            ],
            "stage_four_head_max_abs_error": max_error(manual.stage_logits(4), native),
            "sham_final_max_abs_error": max_error(sham, native),
            "parameter_count": sum(p.numel() for p in self.model.parameters()),
            "dtype": str(self.dtype),
            "attention": "sdpa_math" if self.sdpa else "eager",
            "use_cache": False,
            "token_ids": x.flatten().tolist(),
        }


In [ ]:
%%writefile /kaggle/temp/loop-source/runner.py
"""One isolated GPU worker per checkpoint. Launch via the generated notebook."""
import argparse
import hashlib
import importlib
import json
import os
from pathlib import Path
import sys
import time
import traceback

import torch


def save(path, obj):
    temp = path.with_suffix(f".{os.getpid()}.tmp")
    temp.write_text(json.dumps(obj, indent=2, allow_nan=False) + "\n")
    temp.replace(path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("worker", choices=["ouro", "qwen", "paired"])
    parser.add_argument("--phases", nargs="+", default=["pilot", "heldout"],
                        choices=["pilot", "heldout"])
    parser.add_argument("--tasks", nargs="+", default=["cue", "memory", "carry"],
                        choices=["cue", "memory", "carry"])
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--validate-only", action="store_true")
    args = parser.parse_args()
    out = args.out
    out.mkdir(exist_ok=True, parents=True)
    base = Path(__file__).resolve().parent
    if not torch.cuda.is_available() or not Path("/kaggle/working").is_dir():
        raise RuntimeError("Run this worker in a Kaggle GPU notebook")
    metadata = {
        "platform": "kaggle", "engine": args.worker, "gpu": torch.cuda.get_device_name(),
        "gpu_capacity_bytes": torch.cuda.get_device_properties(0).total_memory,
        "cuda": torch.version.cuda, "torch": str(torch.__version__),
        "visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "code_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                        for p in sorted(base.glob("*.py"))},
    }
    failed = []

    def measure(label, experiment, phase, operation):
        start = time.perf_counter()
        torch.cuda.reset_peak_memory_stats()
        print(f"BEGIN {label}", flush=True)
        try:
            operation()
            torch.cuda.synchronize()
        except Exception:
            error = traceback.format_exc()
            save(out / f"{label}.failure.json", {**metadata, "status": "failed", "error": error})
            print(error, flush=True)
            failed.append(label)
            return False
        save(out / f"{label}.receipt.json", {
            **metadata, "status": "complete", "experiment": experiment, "phase": phase,
            "elapsed_seconds": time.perf_counter() - start,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
        })
        print(f"COMPLETE {label} in {time.perf_counter() - start:.1f}s", flush=True)
        return True

    if args.worker == "paired":
        import paired
        measure("paired-training-control", "training", "control", lambda: paired.run(out))
    else:
        from model_engine import Engine
        engine = Engine.load(args.worker)
        diag = engine.validate()
        save(out / f"{args.worker}-native-validation.json", {**metadata, **diag})
        print(f"Native parity {args.worker}: {diag}", flush=True)
        if args.validate_only:
            return
        passed = {}
        for phase in args.phases:
            for name in args.tasks:
                if phase == "heldout" and passed.get(name) is False:
                    continue
                def run_task():
                    importlib.import_module(name).run(engine, phase, out)
                passed[name] = measure(f"{args.worker}-{name}-{phase}", name, phase, run_task)
    if failed:
        sys.exit(f"Failed workloads: {failed}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/temp/loop-source/cue.py
"""Paired attention cuts for the repeated-clue experiment (one prompt at a time)."""
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np


COLORS = "red blue green black white gold pink gray brown".split()
NAMES = "Amy Ben Dan Emma John Kate Lisa Mark Mary Paul Sam Tom".split()
FRUITS = "pear plum apple orange peach cherry grape lemon lime banana mango fig coconut strawberry".split()
FILLER = " Each locker had a door. One locker had a dent. Each locker had a shelf. One locker had a mark."
PILOT_N, TEST_N = 24, 100
DESIGN = "cue-v1"


@dataclass(frozen=True)
class Scene:
    number: int
    colors: tuple
    fruits: tuple
    people: tuple
    choice: int
    decoy: int
    target_first: bool


def scenes(phase):
    rng = random.Random(20261121)
    items, seen = [], set()
    while len(items) < PILOT_N + TEST_N:
        colors, fruits, people = (tuple(rng.sample(v, 2)) for v in (COLORS, FRUITS, NAMES))
        key = (tuple(sorted(colors)), tuple(sorted(fruits)), tuple(sorted(people)))
        if key in seen:
            continue
        seen.add(key)
        idx = len(items)
        items.append(Scene(idx, colors, fruits, people, rng.randrange(2),
                           rng.randrange(2), idx % 2 == 0))
    if phase == "pilot":
        return items[:PILOT_N]
    if phase == "heldout":
        return items[PILOT_N:]
    raise ValueError(phase)


class Prompt:
    """Sentence-aligned token ranges, checked against whole-string tokenization."""

    def __init__(self, engine, scene, *, reverse=False, clue_first=False):
        choice = scene.choice ^ int(reverse)
        descriptions = ""
        for color, fruit in zip(scene.colors, scene.fruits):
            article = "an" if fruit[0] in "aeiou" else "a"
            descriptions += f" {color.capitalize()} locker holds {article} {fruit}."
        target = f" {scene.people[0]} uses the {scene.colors[choice]} locker."
        sham = f" {scene.people[1]} uses the {scene.colors[scene.decoy]} locker."
        clues = [("target", target), ("sham", sham)]
        if not scene.target_first:
            clues.reverse()
        parts = [("descriptions", descriptions)]
        parts = clues + parts if clue_first else parts + clues
        self.record = "".join(text for _, text in parts)
        question = f" Which fruit is in {scene.people[0]}'s locker? Answer:"
        self.text = self.record * 2 + question
        self.ids = []
        self.first = {}
        for name, part in parts:
            start = len(self.ids)
            self.ids.extend(engine.ids(part))
            self.first[name] = (start, len(self.ids))
        self.split = len(self.ids)
        self.ids = self.ids * 2 + engine.ids(question)
        assert self.ids == engine.ids(self.text)
        a, b = self.first["descriptions"]
        self.descriptions = (a + self.split, b + self.split)
        a, b = self.first["target"]
        c, d = self.first["sham"]
        assert b - a == d - c
        correct, wrong = scene.fruits[choice], scene.fruits[1 - choice]
        self.answer_words = (correct, wrong)
        ids = []
        for fruit in self.answer_words:
            continuation = engine.ids(" " + fruit)
            assert len(continuation) == 1 and engine.ids(self.text + " " + fruit) == self.ids + continuation
            ids.append(continuation[0])
        self.answers = tuple(ids)


def cut_mask(queries, keys, sites=None):
    """Mask exactly the stated query rows and key columns in all heads."""
    q0, q1 = queries
    k0, k1 = keys
    assert 0 <= k0 < k1 <= q0 < q1

    def apply(stage, offset, mask):
        if sites is None or (stage, offset) in sites:
            mask[..., q0:q1, k0:k1] = float("-inf")
        return mask

    return apply


def score(engine, prompt, ids=None, mask_fn=None):
    import torch

    logits = engine.run(prompt.ids if ids is None else ids, mask_fn=mask_fn,
                        logits_positions=[-1]).logits[0, -1].float()
    if not torch.isfinite(logits).all():
        raise ValueError("Nonfinite final logits")
    a, b = prompt.answers
    probs = logits.softmax(0)
    top = int(logits.argmax())
    top_text = engine.tokenizer.decode([top], skip_special_tokens=False)
    top_word = top_text.strip().casefold()
    return dict(margin=float(logits[a] - logits[b]), correct_probability=float(probs[a]),
                other_probability=float(probs[b]), top_token=top, top_text=top_text,
                top_correct=top_word == prompt.answer_words[0],
                off_candidate=top_word not in prompt.answer_words)


def isolation_check(engine, prompt):
    """Offset-matched isolated suffix and same-shape, changed-prefix leakage check."""
    import torch

    split = prompt.split
    close = cut_mask((split, len(prompt.ids)), (0, split))
    def final(ids, *, mask=None, offset=0):
        return engine.run(ids, mask_fn=mask, position_offset=offset,
                          logits_positions=[-1]).logits[0, -1].float()
    closed = final(prompt.ids, mask=close)
    isolated = final(prompt.ids[split:], offset=split)
    replacement = list(reversed(prompt.ids[:split])) + prompt.ids[split:]
    assert replacement != prompt.ids
    changed = final(replacement, mask=close)
    isolated_err = (closed - isolated).abs()
    leak = float((changed - closed).abs().max())
    metrics = dict(position_offset=split, altered_prefix_max_abs=leak,
                   isolated_max_abs=float(isolated_err.max()),
                   isolated_rms=float(isolated_err.square().mean().sqrt()),
                   isolated_rms_limit=0.025, isolated_max_limit=0.25,
                   altered_prefix_limit=0.002, vocabulary_size=len(closed))
    if not (torch.isfinite(isolated_err).all() and metrics["isolated_max_abs"] <= 0.25
            and metrics["isolated_rms"] <= 0.025 and leak <= 0.002):
        raise AssertionError(f"Failed cross-copy isolation: {metrics}")
    return metrics


def run_scene(engine, scene, primary, neighbors):
    p = Prompt(engine, scene)
    def cut(key, visits=None):
        return score(engine, p, mask_fn=cut_mask(p.descriptions, p.first[key], visits))
    early = (2, primary)
    late = (3, primary)
    s = {"intact": score(engine, p)}
    for key in ("target", "sham"):
        for label, visits in (("early", {early}), ("late", {late}), ("both", {early, late})):
            s[key + "_" + label] = cut(key, visits)
    s["target_all_layers"] = cut("target")
    for offset in neighbors:
        site = (3, offset)
        s[f"neighbor_{offset}_late"] = cut("target", {site})
        s[f"neighbor_{offset}_both"] = cut("target", {early, site})
    s["reversed_assignment"] = score(engine, Prompt(engine, scene, reverse=True))
    s["one_copy"] = score(engine, p, ids=p.ids[p.split:])
    filler_ids = engine.ids(FILLER)
    assert len(filler_ids) == p.split and filler_ids + p.ids[p.split:] == engine.ids(FILLER + p.text[len(p.record):])
    s["filler"] = score(engine, p, ids=filler_ids + p.ids[p.split:])
    s["clue_first"] = score(engine, Prompt(engine, scene, clue_first=True))
    return dict(scene=asdict(scene), prompt=p.text, answers=list(zip(p.answer_words, p.answers)),
                split=p.split, copy1=p.first, copy2_queries=p.descriptions, scores=s)


def interaction(m, early, late, both):
    return (m[early] - m[both]) - (m["intact"] - m[late])


def scene_metrics(row, neighbor):
    scores = row["scores"]
    m = {k: v["margin"] for k, v in scores.items()}
    i = interaction(m, "target_early", "target_late", "target_both")
    sham = interaction(m, "sham_early", "sham_late", "sham_both")
    nearby = interaction(m, "target_early", f"neighbor_{neighbor}_late", f"neighbor_{neighbor}_both")
    metrics = dict(target_interaction=i, sham_interaction=sham, nearby_interaction=nearby,
                   target_minus_sham=i - sham, target_minus_nearby=i - nearby,
                   early_cost=m["intact"] - m["target_early"], late_cost=m["intact"] - m["target_late"],
                   late_cost_after_early=m["target_early"] - m["target_both"],
                   all_layers_cost=m["intact"] - m["target_all_layers"],
                   duplication_gain=m["intact"] - m["one_copy"], evidence_vs_filler=m["intact"] - m["filler"],
                   clue_order_gain=m["clue_first"] - m["intact"],
                   correct_both_assignments=float(m["intact"] > 0 and m["reversed_assignment"] > 0),
                   nearby_single_cut_gap=m[f"neighbor_{neighbor}_late"] - m["target_late"])
    for condition, v in scores.items():
        for k in ("margin", "correct_probability", "other_probability", "top_correct", "off_candidate"):
            metrics[f"{condition}.{k}"] = float(v[k])
        metrics[f"{condition}.candidate_accuracy"] = float(v["margin"] > 0)
        metrics[f"{condition}.tie"] = float(v["margin"] == 0)
    return metrics


def intervals(rows):
    rng = np.random.default_rng(20261122)
    idx = rng.integers(len(rows), size=(4000, len(rows)))
    result = {}
    for key in rows[0]:
        values = np.array([r[key] for r in rows], dtype=float)
        low, high = np.percentile(values[idx].mean(axis=1), [2.5, 97.5])
        result[key] = dict(mean=float(values.mean()), lo=float(low), hi=float(high))
    return result


def write_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, allow_nan=False, ensure_ascii=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def pick_neighbor(rows, neighbors):
    """Match the single late-cut cost on pilot means before evaluating held-out scenes."""
    gaps = {}
    for offset in neighbors:
        diffs = [r["scores"][f"neighbor_{offset}_late"]["margin"] - r["scores"]["target_late"]["margin"]
                 for r in rows]
        gaps[offset] = float(np.mean(diffs))
    return min(neighbors, key=lambda o: (abs(gaps[o]), o)), gaps


def plot(estimates, path, title):
    from matplotlib import pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.7), constrained_layout=True)
    for ax, keys, labels, heading in (
            (axes[0], [x + ".margin" for x in ("intact", "target_early", "target_late", "target_both")],
             ["Intact", "Stage 2", "Stage 3", "Both"], "Final correct-minus-other logit"),
            (axes[1], ["target_interaction", "sham_interaction", "nearby_interaction"],
             ["Clue", "Sham", "Nearby offset"], "Paired-cut interaction (logits)")):
        stats = [estimates[key] for key in keys]
        ax.axvline(0, color="0.7", linewidth=1)
        for j, stat in enumerate(stats):
            ax.plot([stat["lo"], stat["hi"]], [j, j], color="#326897")
            ax.plot(stat["mean"], j, "o", color="#19466e")
        ax.set_yticks(range(len(labels)), labels)
        ax.invert_yaxis()
        ax.set_title(heading, fontsize=10)
    fig.suptitle(title)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def run(engine, phase, outdir):
    out = Path(outdir) / "cue"
    out.mkdir(parents=True, exist_ok=True)
    model = engine.name
    path = out / f"{model}_{phase}_rows.json"
    cfg_path = out / f"{model}_frozen_sites.json"
    items = scenes(phase)
    primary = (engine.width + 1) // 2
    delta = max(1, round(engine.width / 8))
    candidates = [primary - delta, primary + delta]
    if phase == "heldout":
        cfg = json.loads(cfg_path.read_text())
        assert (cfg["design"], cfg["primary"], cfg["width"]) == (DESIGN, primary, engine.width)
        neighbors = [cfg["late_neighbor"]]
    else:
        neighbors = candidates
    records = json.loads(path.read_text()) if path.exists() else {"design": DESIGN, "rows": []}
    rows = records["rows"]
    assert records["design"] == DESIGN
    assert [r["scene"]["number"] for r in rows] == [x.number for x in items[:len(rows)]]
    if "isolation" not in records:
        records["isolation"] = isolation_check(engine, Prompt(engine, items[0]))
        write_json(path, records)
    for item in items[len(rows):]:
        rows.append(run_scene(engine, item, primary, neighbors))
        write_json(path, records)
        if len(rows) % 10 == 0 or len(rows) == len(items):
            print(f"Repeated clue {model} {phase}: {len(rows)}/{len(items)}", flush=True)
    # Verify exact balance of clue length, key position, and key-to-query distance.
    def geometry(row, key):
        a, b = row["copy1"][key]
        c, d = row["copy2_queries"]
        return (a, b, c, d)
    assert sorted(geometry(r, "target") for r in rows) == sorted(geometry(r, "sham") for r in rows)
    if phase == "pilot":
        neighbor, gaps = pick_neighbor(rows, neighbors)
        cfg = dict(design=DESIGN, model=model, width=engine.width, primary=primary,
                   late_neighbor=neighbor, pilot_single_cut_margin_differences=gaps,
                   selection="Smallest absolute difference in mean single-cut margin on pilot scenes")
        write_json(cfg_path, cfg)
    neighbor = cfg["late_neighbor"]
    estimates = intervals([scene_metrics(r, neighbor) for r in rows])
    summary = dict(design=DESIGN, model=model, phase=phase, n_families=len(rows), sites=cfg,
                   executed_blocks_1_based=[engine.width + primary, 2 * engine.width + primary],
                   alternate_late_block_1_based=2 * engine.width + neighbor,
                   isolation=records["isolation"], estimates=estimates,
                   bootstrap="4000 resamples of all scene families; percentile 95% intervals",
                   off_candidate="Decoded greedy next token falls outside the two named fruits; "
                                 "case and surrounding whitespace ignored")
    write_json(out / f"{model}_{phase}_summary.json", summary)
    plot(estimates, out / f"{model}_{phase}.png", f"{model} · repeated clue · {phase} (n={len(rows)})")
    if phase == "heldout" and model in ("qwen", "ouro"):
        paths = [out / f"{tag}_heldout_rows.json" for tag in ("ouro", "qwen")]
        if all(p.exists() for p in paths):
            datasets = [json.loads(p.read_text())["rows"] for p in paths]
            if all(len(ds) == TEST_N for ds in datasets):
                sites = [json.loads((out / f"{tag}_frozen_sites.json").read_text())["late_neighbor"]
                         for tag in ("ouro", "qwen")]
                differences = []
                common_success = []
                for ouro, qwen in zip(*datasets):
                    assert ouro["scene"] == qwen["scene"] and ouro["prompt"] == qwen["prompt"]
                    a, b = scene_metrics(ouro, sites[0]), scene_metrics(qwen, sites[1])
                    diff = {k: a[k] - b[k] for k in ("target_interaction", "target_minus_sham",
                            "target_minus_nearby", "duplication_gain", "intact.candidate_accuracy")}
                    differences.append(diff)
                    if a["intact.candidate_accuracy"] and b["intact.candidate_accuracy"]:
                        common_success.append(diff)
                write_json(out / "heldout_ouro_minus_qwen.json", dict(
                    n=TEST_N, estimates=intervals(differences), secondary_common_clean_success=dict(
                        n=len(common_success), estimates=intervals(common_success) if common_success else {})))
    return summary


In [ ]:
%%writefile /kaggle/temp/loop-source/memory.py
"""Source-closed memory, witness counterfactuals and matched register controls."""
import itertools
import json
import random
from pathlib import Path

import numpy as np
import torch


BOXES = tuple("ABCDEFGHIJKL")
SLOTS = 8
QUESTION_WIDTH = 16
ROLES = ("first", "second", "world")
DOMAINS = ("people", "registers")
PHASE_SIZES = {"pilot": 24, "heldout": 120}
SEED = 170271
BOOTSTRAPS = 2000


def families(phase):
    rng = random.Random(SEED)
    layouts = list(itertools.combinations(BOXES, 3))
    rng.shuffle(layouts)
    items = []
    for index, layout in enumerate(layouts[:sum(PHASE_SIZES.values())]):
        boxes = list(layout)
        rng.shuffle(boxes)
        items.append((index, boxes, index % 2))
    n = PHASE_SIZES["pilot"]
    return items[:n] if phase == "pilot" else items[n:]


def facts(domain, boxes, witness, toggle):
    a, b, c = boxes
    watching = [witness == index or toggle for index in range(2)]
    if domain == "people":
        actors = ["Alice", "Bob"]
        rule = "People remember the last box they saw the coin in."
        start = f"The coin is in box {a}. Alice and Bob both see it."
        mid = " ".join(f"{name} sees this move: {'yes' if sees else 'no'}."
                       for name, sees in zip(actors, watching))
        end = "Neither person sees this move."
    else:
        actors = ["register A", "register B"]
        rule = "Registers store the last box they recorded the coin in."
        start = f"The coin is in box {a}. Both registers record its location."
        mid = " ".join(f"{name.capitalize()} records this move: {'yes' if sees else 'no'}."
                       for name, sees in zip(actors, watching))
        end = "Neither register records this move."
    return f"{rule}\n{start}\nThe coin moves to box {b}. {mid}\nThe coin moves to box {c}. {end}\n"


def question(domain, role):
    if role == "world":
        ask = "Where is the coin actually located now?"
    elif domain == "people":
        person = ("Alice", "Bob")[ROLES.index(role)]
        ask = f"Where does {person} believe the coin is?"
    else:
        register = ("A", "B")[ROLES.index(role)]
        ask = f"Which location is stored in register {register}?"
    return f"\nQuestion: {ask}\nAnswer: box"


def mask_for(mode, n_source):
    n_question = n_source + SLOTS

    def mask_fn(stage, offset, causal):
        if mode == "sham":
            return causal
        mask = causal
        blocked = torch.finfo(mask.dtype).min
        if mode == "closed" or stage > 2:
            mask[:, :, n_source:, :n_source] = blocked
        if stage <= 2:
            mask[:, :, n_question:, :n_question] = blocked
        return mask

    return mask_fn


def replace_memory(engine, n_source, memory):
    def edit(stage, offset, hidden):
        if (stage, offset) == (2, engine.width):
            hidden = hidden.clone()
            hidden[:, n_source:n_source + SLOTS] = memory
        return hidden
    return edit


def question_branch(source, slots, q):
    """Place all right padding strictly after the position read by the output head."""
    if len(q) > QUESTION_WIDTH:
        raise ValueError(f"Question length {len(q)} exceeds the frozen width {QUESTION_WIDTH}")
    end = len(source) + len(slots) + len(q) - 1
    ids = source + slots + q + [slots[0]] * (QUESTION_WIDTH - len(q))
    return ids, end


def read_memory(engine, source, slots):
    ids, end = question_branch(source, slots, [])
    key = (2, engine.width)
    result = engine.run(ids, capture=(key,), mask_fn=mask_for("memory", len(source)), logits_positions=[end])
    return result.states[key][:, len(source):len(source) + SLOTS].detach().clone()


def close_enough(left, right, label):
    left, right = left.float(), right.float()
    delta = (left - right).abs()
    if not torch.isfinite(delta).all().item():
        raise AssertionError(f"{label}: non-finite values")
    # Keep the predeclared FP16 tolerance for all same-shape comparisons.
    if not torch.allclose(left, right, rtol=0.002, atol=0.015625):
        raise AssertionError(f"{label}: max difference {float(delta.max())}")
    return float(delta.max())


def controls(engine, source, slots, q, alt_q, other, memory):
    ids, end = question_branch(source, slots, q)
    alt_ids, alt_end = question_branch(source, slots, alt_q)
    other_ids, _ = question_branch(other, slots, q)
    n = len(source)
    if len(other) != n:
        raise ValueError("The leak test requires facts with identical token lengths")
    if len(q) != len(alt_q) or q == alt_q:
        raise ValueError("Memory invariance requires distinct, equally long questions")
    sham = engine.run(ids, mask_fn=mask_for("sham", n), logits_positions=[end]).logits
    native = engine.run(ids, logits_positions=[end]).logits
    sham_diff = close_enough(sham, native, "sham versus native")
    del sham, native
    blocked = mask_for("closed", n)
    old = engine.run(ids, mask_fn=blocked, logits_positions=[end]).logits
    new = engine.run(other_ids, mask_fn=blocked, logits_positions=[end]).logits
    closed_diff = close_enough(old, new, "changing facts with all routes closed")
    del old, new
    key = (2, engine.width)
    masked = mask_for("memory", n)
    run = engine.run(ids, mask_fn=masked, capture=(key,), logits_positions=[end])
    alternate = engine.run(alt_ids, mask_fn=masked, capture=(key,), logits_positions=[alt_end])
    first = run.states[key][:, n:n + SLOTS]
    second = alternate.states[key][:, n:n + SLOTS]
    memory_diff = close_enough(memory, first, "neutral versus real question at the same shape")
    alt_diff = close_enough(first, second, "changing an equal-length question")
    del run, alternate, first, second
    # Reuse exactly the same boundary tensor while changing the facts.
    edit = replace_memory(engine, n, memory)
    before = engine.run(ids, mask_fn=masked, edit_fn=edit, logits_positions=[end]).logits
    after = engine.run(other_ids, mask_fn=masked, edit_fn=edit, logits_positions=[end]).logits
    fixed_diff = close_enough(before, after, "fixed memory, changed source")
    return dict(sham_max_abs=sham_diff, source_closed_max_abs=closed_diff,
                memory_independence_max_abs=memory_diff, changed_question_max_abs=alt_diff,
                fixed_memory_max_abs=fixed_diff)


def score(engine, logits, answer_ids, boxes, target):
    logits = logits[0, -1].detach().float()
    scores = logits[answer_ids]
    probs = (scores - torch.logsumexp(logits, dim=0)).exp().tolist()
    values = scores.tolist()
    if not np.isfinite(values + probs).all():
        raise AssertionError("Non-finite answer scores")
    options = [BOXES.index(box) for box in boxes]
    best = max(options, key=values.__getitem__)
    greedy = int(logits.argmax())
    return dict(letter_logits=values, winner=BOXES[best], greedy_id=greedy,
                greedy_text=engine.tokenizer.decode([greedy]), malformed=greedy not in answer_ids,
                outside_scene=greedy not in [answer_ids[i] for i in options],
                free_correct=greedy == answer_ids[BOXES.index(target)],
                answer_mass=sum(probs[i] for i in options), correct_mass=probs[BOXES.index(target)])


def ask(engine, source, slots, q_ids, memory, answer_ids, boxes, target):
    kwargs = {}
    if memory is not None:
        kwargs.update(mask_fn=mask_for("memory", len(source)),
                      edit_fn=replace_memory(engine, len(source), memory))
    ids, end = question_branch(source, slots, q_ids)
    result = engine.run(ids, capture=((3, engine.width),), logits_positions=[end], **kwargs)
    return {stage: score(engine, logits, answer_ids, boxes, target)
            for stage, logits in [(3, result.stage_logits(3, positions=[end])), (4, result.logits)]}


def collect(engine, phase, directory, slots, answer_ids):
    layouts = families(phase)
    for family, boxes, witness in layouts:
        path = directory / f"scene-{family:03d}.json"
        if path.exists() and json.loads(path.read_text()).get("question_width") == QUESTION_WIDTH:
            continue
        rows = []
        checks = {}
        # A different scene cyclically reassigns identical box names to events.
        # This derangement keeps the answer vocabulary and event permissions fixed.
        foil = boxes[1:] + boxes[:1]
        for domain in DOMAINS:
            for toggle in (False, True):
                source = engine.ids(facts(domain, boxes, witness, toggle))
                other = engine.ids(facts(domain, foil, witness, toggle))
                if len(source) != len(other):
                    raise AssertionError("Shuffled scene has a different position offset")
                mem = read_memory(engine, source, slots)
                memories = {"direct": None, "memory": mem}
                if not toggle:
                    memories["shuffled"] = read_memory(engine, other, slots)
                if family == layouts[0][0] and not toggle:
                    checks[domain] = controls(engine, source, slots, engine.ids(question(domain, "world")),
                                              engine.ids(question(domain, "first")), other, mem)
                for role_index, role in enumerate(ROLES):
                    q_ids = engine.ids(question(domain, role))
                    target_index = 2 if role == "world" else int(role_index == witness or toggle)
                    target = boxes[target_index]
                    for condition, selected in memories.items():
                        scores = ask(engine, source, slots, q_ids, selected, answer_ids, boxes, target)
                        for stage, scores_at_stage in scores.items():
                            rows.append(dict(family=family, boxes=boxes, witness=witness, domain=domain,
                                             toggle=toggle, role=role, condition=condition, stage=stage,
                                             target=target, donor_target=foil[target_index], **scores_at_stage))
        atomic(path, {"rows": rows, "checks": checks, "question_width": QUESTION_WIDTH})
        print(f"memory {engine.name} {phase}, scene {family} complete", flush=True)


def atomic(path, obj):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(obj, allow_nan=False), encoding="utf-8")
    temp.replace(path)


def read_rows(directory):
    rows = []
    for path in sorted(directory.glob("scene-*.json")):
        scene = json.loads(path.read_text())
        if scene.get("question_width") != QUESTION_WIDTH:
            raise ValueError(f"Rerun {path.parent} with fixed question width before calibration")
        rows.extend(scene["rows"])
    return rows


def calibrate(train, evaluate):
    """Identical ridge readouts on all twelve box logits; train only on calibration scenes."""
    for stage in (3, 4):
        tr = [r for r in train if r["stage"] == stage and r["condition"] in ("memory", "direct")]
        te = [r for r in evaluate if r["stage"] == stage]
        x = np.asarray([r["letter_logits"] for r in tr])
        y = np.eye(len(BOXES))[[BOXES.index(r["target"]) for r in tr]]
        mean, std = x.mean(0), np.maximum(x.std(0), 1e-5)
        x = (x - mean) / std
        intercept = y.mean(0)
        beta = np.linalg.solve(x.T @ x + len(x) * 0.05 * np.eye(len(BOXES)), x.T @ (y - intercept))
        for r in te:
            pred = (np.asarray(r["letter_logits"]) - mean) / std @ beta + intercept
            r["calibrated_winner"] = max(r["boxes"], key=lambda b: pred[BOXES.index(b)])


def interval(family_values):
    vals = np.array(family_values, dtype=float)
    rng = np.random.default_rng(SEED)
    samples = rng.choice(vals, (BOOTSTRAPS, len(vals)), replace=True).mean(1)
    return dict(mean=float(vals.mean()), ci95=np.quantile(samples, [0.025, 0.975]).tolist(), families=len(vals))


def summarize(rows):
    result, contrasts = {}, {}
    families_in_run = sorted({r["family"] for r in rows})
    by = {(r["family"], r["domain"], r["toggle"], r["condition"], r["role"], r["stage"]): r for r in rows}
    for domain, toggle, cond, stage in itertools.product(DOMAINS, (False, True),
                                                        ("memory", "direct", "shuffled"), (3, 4)):
        if toggle and cond == "shuffled":
            continue
        scene_rows = [[by[f, domain, toggle, cond, role, stage] for role in ROLES] for f in families_in_run]
        key = f"{domain}/{'both' if toggle else 'one'}/{cond}/stage{stage}"
        metrics = {}
        for field, label in (("winner", "native"), ("calibrated_winner", "calibrated")):
            scores = [[r[field] == r["target"] for r in scene] for scene in scene_rows]
            metrics[label + "_mean_accuracy"] = interval(np.mean(scores, axis=1))
            metrics[label + "_all_three"] = interval(np.all(scores, axis=1))
            matrix = [[sum(r[field] == r["boxes"][col] for scene in scene_rows for r in [scene[i]])
                       for col in range(3)] for i in range(3)]
            metrics[label + "_event_confusion_counts"] = matrix
        for attr in ("answer_mass", "correct_mass", "malformed", "outside_scene", "free_correct"):
            metrics[attr] = interval([np.mean([r[attr] for r in scene]) for scene in scene_rows])
        result[key] = metrics
    for readout in ("winner", "calibrated_winner"):
        drifts = {}
        for domain in DOMAINS:
            shifts, selective, recall, donor, rightful = [], [], [], [], []
            for f in families_in_run:
                get = lambda cond, role, stage, both=False: by[f, domain, both, cond, role, stage]
                early = [get("memory", r, 3) for r in ROLES[:2]]
                late = [get("memory", r, 4) for r in ROLES[:2]]
                shifts.append(np.mean([int(v[readout] == v["boxes"][2]) - int(u[readout] == u["boxes"][2])
                                       for u, v in zip(early, late)]))
                orig = [get("memory", r, 4) for r in ROLES]
                switched = [get("memory", r, 4, True) for r in ROLES]
                affected = 1 - orig[0]["witness"]
                selective.append(all((a[readout] == a["target"] and b[readout] == b["target"])
                                     if idx == affected else (a[readout] == b[readout])
                                     for idx, (a, b) in enumerate(zip(orig, switched))))
                world, shuffle = orig[2], get("shuffled", "world", 4)
                recall.append(world[readout] == world["target"])
                donor.append(shuffle[readout] == shuffle["donor_target"])
                rightful.append(int(recall[-1]) - int(shuffle[readout] == shuffle["target"]))
            drifts[domain] = np.asarray(shifts)
            contrasts[f"{domain}/{readout}"] = dict(late_world_intrusion_change=interval(shifts),
                witness_only_counterfactual_correct=interval(selective), own_world_accuracy=interval(recall),
                shuffled_donor_world_accuracy=interval(donor), own_minus_shuffled_world_accuracy=interval(rightful))
        contrasts[f"people_minus_registers/{readout}"] = interval(drifts["people"] - drifts["registers"])
    return {"metrics": result, "paired_contrasts": contrasts}


def plots(summary, output):
    import matplotlib.pyplot as plt
    for method in ("native", "calibrated"):
        fig, axes = plt.subplots(2, 3, figsize=(9, 6), gridspec_kw={"width_ratios": [1.4, 1, 1]})
        for i, domain in enumerate(DOMAINS):
            axes[i, 0].axis("off")
            text = ("People\nA coin goes from box 1 to 2 to 3.\nBoth people see its placement.\nOnly one sees the first move.\nNeither sees the second."
                    if i == 0 else "Registers\nBoth record the initial box.\nOnly one records the first move.\nNeither records the second.")
            axes[i, 0].text(0, .7, text, va="top", fontsize=9)
            for j, stage in enumerate((3, 4), 1):
                ax = axes[i, j]
                matrix = np.array(summary["metrics"][f"{domain}/one/memory/stage{stage}"][method + "_event_confusion_counts"])
                freq = matrix / matrix.sum(axis=1, keepdims=True)
                ax.imshow(freq, vmin=0, vmax=1, cmap="Blues")
                for y, x in itertools.product(range(3), repeat=2):
                    ax.text(x, y, f"{freq[y,x]:.0%}", ha="center", va="center",
                            color="white" if freq[y,x] > .55 else "black")
                ax.set_xticks(range(3), ["initial", "middle", "final"], fontsize=8)
                ax.set_yticks(range(3), ["Alice", "Bob", "world"] if i == 0 else ["reg. A", "reg. B", "world"])
                ax.set_title(f"Stage {stage}")
        fig.tight_layout()
        fig.savefig(output / f"{method}-confusion.png", dpi=140)
        plt.close(fig)


def rooms(engine, phase, slots, answer_ids):
    rows = []
    for family, boxes, witness in families(phase):
        rooms_and_boxes = dict(zip(("red", "blue"), boxes[:2]))
        sources, memories = {}, {}
        for color, box in rooms_and_boxes.items():
            source = engine.ids(f"In the {color} room, the key is in box {box}.\n")
            sources[color] = source
            memories[color] = read_memory(engine, source, slots)
        if len(sources["red"]) != len(sources["blue"]):
            raise AssertionError("Room memory offsets differ")
        blend = ((memories["red"].float() + memories["blue"].float()) * 0.5).to(memories["red"].dtype)
        for color, target in rooms_and_boxes.items():
            q = engine.ids(f"\nQuestion: Where is the key in the {color} room?\nAnswer: box")
            for condition, memory in ((color + "_alone", memories[color]), ("blend", blend)):
                measured = ask(engine, sources[color], slots, q, memory, answer_ids, boxes[:2], target)[4]
                rows.append(dict(family=family, color=color, target=target, condition=condition, **measured))
    return rows


@torch.no_grad()
def run(engine, phase, outdir):
    if phase not in PHASE_SIZES:
        raise ValueError("phase must be 'pilot' or 'heldout'")
    directory = Path(outdir) / "memory" / engine.name / phase
    directory.mkdir(parents=True, exist_ok=True)
    caps = [engine.ids(" " + box) for box in BOXES]
    neutral = engine.ids(" .")
    if any(len(seq) != 1 for seq in [*caps, neutral]) or len(set(seq[0] for seq in caps)) != len(BOXES):
        raise ValueError(f"Tokenizer must supply distinct, single-token box answers and neutral slots: {caps}, {neutral}")
    slots, answer_ids = neutral * SLOTS, [seq[0] for seq in caps]
    collect(engine, phase, directory, slots, answer_ids)
    rows = read_rows(directory)
    if phase == "pilot":
        for fold in range(4):
            calibrate([r for r in rows if r["family"] % 4 != fold],
                      [r for r in rows if r["family"] % 4 == fold])
    else:
        training = read_rows(directory.parent / "pilot")
        if len({r["family"] for r in training}) != PHASE_SIZES["pilot"]:
            raise ValueError("Need complete pilot calibration rows from the same engine before heldout")
        calibrate(training, rows)
    summary = summarize(rows)
    summary.update(checkpoint=engine.name, phase=phase, sizes=PHASE_SIZES,
                   answer_ids=dict(zip(BOXES, answer_ids)), slot_ids=slots, question_width=QUESTION_WIDTH, seed=SEED,
                   readout="Twelve box logits to twelve box scores, ridge penalty 0.05, per stage/model; pilot fourfold grouped crossfit.",
                   boundary_blocks=[2 * engine.width, 3 * engine.width, 4 * engine.width],
                   example={"facts": facts("people", ["A", "B", "C"], 1, False),
                            "question": question("people", "first")})
    gate_rule = "Pilot world-only own and shuffled-to-donor accuracy lower 95% bounds > 1/3, and own minus shuffled-to-recipient lower bound > 0."
    if phase == "pilot":
        main = summary["paired_contrasts"]["people/winner"]
        gate = (main["own_world_accuracy"]["ci95"][0] > 1/3
                and main["shuffled_donor_world_accuracy"]["ci95"][0] > 1/3
                and main["own_minus_shuffled_world_accuracy"]["ci95"][0] > 0)
    else:
        gate = json.loads((directory.parent / "pilot" / "summary.json").read_text())["room_gate"]["passed"]
    summary["room_gate"] = {"rule": gate_rule, "passed": bool(gate)}
    if gate:
        room_rows = rooms(engine, phase, slots, answer_ids)
        atomic(directory / "room-rows.json", room_rows)
        summary["room_blend"] = {f"{color}/{condition}": interval(
            [r["winner"] == r["target"] for r in room_rows if r["color"] == color and r["condition"] == condition])
            for color in ("red", "blue") for condition in (color + "_alone", "blend")}
    atomic(directory / "summary.json", summary)
    plots(summary, directory)
    return summary


In [ ]:
%%writefile /kaggle/temp/loop-source/carry.py
"""Carry-directed residual swaps on matched two-digit addition problems."""

from collections import defaultdict
from dataclasses import dataclass
import hashlib
import json
import math
from pathlib import Path
import random
import re

import numpy as np
import torch


WORDS = {
    "English": "zero one two three four five six seven eight nine".split(),
    "French": "zéro un deux trois quatre cinq six sept huit neuf".split(),
}
LANGUAGES = tuple(WORDS)
ALL_WORDS = tuple(dict.fromkeys(word for words in WORDS.values() for word in words))
INSTRUCTION = (
    "Write the tens digit of the sum as one word in the requested language.\n"
    "14 + 18 | English: three\n23 + 21 | French: quatre\n"
)
# Different totals, even across reader fitting, pilot and held-out splits.
UNITS = {"train": [0, 2, 8], "pilot": [1, 7], "heldout": [3, 4, 5, 6]}
SIZES = {"train": 36, "pilot": 24, "heldout": 100}
SEEDS = {"train": 9187, "pilot": 9188, "heldout": 9189}


def families(split):
    rng = random.Random(SEEDS[split])
    pools = {}
    for tens in range(3, 9):
        for unit in UNITS[split]:
            total = 10 * tens + unit
            if total in (32, 44):  # The two worked examples in the fixed prompt.
                continue
            groups = [[], []]
            for a in range(10, total // 2 + 1):
                b = total - a
                if a == b or b > 99:
                    continue
                groups[int(a % 10 + b % 10 >= 10)].append((a, b))
            for group in groups:
                rng.shuffle(group)
            if min(map(len, groups)) >= 2:
                # Reserve same-carry sham operands separate from all clean operands.
                # The secondary total-clustered interval accounts for reused shams.
                shams = [[group.pop() for _ in range(max(1, len(group)//5))]
                         for group in groups]
                pools[total] = groups, shams
    items = []
    # Use totals uniformly round-robin, then reuse a total with new operands.
    while len(items) < SIZES[split]:
        totals = list(pools)
        rng.shuffle(totals)
        previous = len(items)
        for total in totals:
            groups, reserved = pools[total]
            if min(map(len, groups)) < 1:
                continue
            def order(pair):
                return pair if rng.randrange(2) else pair[::-1]
            clean, sham = [], []
            for group, options in zip(groups, reserved):
                clean.append(order(group.pop()))
                sham.append(order(rng.choice(options)))
            items.append(dict(index=len(items), total=total, sums=clean, sham=sham))
            if len(items) == SIZES[split]:
                break
        if previous == len(items):
            raise ValueError(f"Insufficient distinct operands in {split}")
    return items


@dataclass
class Prompt:
    text: str
    ids: list
    positions: dict


def make_prompt(engine, operands, language):
    a, b = operands
    text = f"{INSTRUCTION}{a} + {b} | {language}:"
    encoded = engine.tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(encoded["input_ids"])
    offsets = encoded["offset_mapping"]
    def token_at(char):
        return next(i for i, (start, end) in enumerate(offsets) if start <= char < end)
    positions = {
        "left operand": token_at(len(INSTRUCTION) + len(str(a)) - 1),
        "right operand": token_at(len(INSTRUCTION) + len(f"{a} + {b}") - 1),
        "answer position": len(ids) - 1,
    }
    return Prompt(text, ids, positions)


class Scorer:
    """Score every token of each word. Share forwards with the same token prefix."""

    def __init__(self, engine, prompt):
        self.engine = engine
        self.prompt = prompt
        self.paths = {}
        for word in ALL_WORDS:
            whole = engine.ids(prompt.text + " " + word)
            if whole[:len(prompt.ids)] != prompt.ids:
                raise ValueError(f"Unstable answer boundary for {word!r}")
            suffix = tuple(whole[len(prompt.ids):])
            if not suffix:
                raise ValueError(f"Empty continuation for {word!r}")
            self.paths[word] = suffix
        if len(set(self.paths.values())) != len(self.paths):
            raise ValueError("Distinct answer strings collapse under the tokenizer")

    def scores(self, words, first_logits, edit=None):
        logits = {(): first_logits}
        def distribution(prefix):
            if prefix not in logits:
                logits[prefix] = self.engine.run(
                    self.prompt.ids + list(prefix), edit_fn=edit, logits_positions=[-1]
                ).logits[0, -1].detach().float()
            return logits[prefix]
        distributions = {}
        scores = {}
        for word in dict.fromkeys(words):
            sequence = self.paths[word]
            score = 0.0
            for i, tok in enumerate(sequence):
                prefix = sequence[:i]
                if prefix not in distributions:
                    distributions[prefix] = distribution(prefix).log_softmax(-1)
                score += distributions[prefix][tok].item()
            scores[word] = score

        generated = []
        ended = False
        word = ""
        text = ""
        eos = self.engine.tokenizer.eos_token_id
        for _ in range(7):
            tok = int(distribution(tuple(generated)).argmax().item())
            if tok == eos:
                ended = True
                break
            generated.append(tok)
            text = self.engine.tokenizer.decode(generated, skip_special_tokens=False)
            # A boundary following the first word makes its identity unambiguous.
            match = re.match(r"^\s*([^\W_]+|[^\w\s])", text, re.UNICODE)
            if match:
                word = match.group(1).casefold()
                if match.end() < len(text) or not word[0].isalnum():
                    ended = True
                    break
        if not ended:
            word = ""  # A truncated word cannot establish the response.
        labels = [language for language, vocab in WORDS.items() if word in vocab]
        digit = next((WORDS[language].index(word) for language in labels), None)
        if digit is None and len(word) == 1 and word in "0123456789":
            digit = int(word)
        return dict(scores=scores, answer_text=text, first_word=word,
                    digit=digit, languages=labels, malformed=not labels,
                    terminated=ended, best_word=max(scores, key=scores.get))

    def gradients(self, target, original, sites, positions):
        # Linearize the very same full-continuation log-probability margin used
        # in the intervened runs, including later conditional token probabilities.
        coefficients = defaultdict(lambda: defaultdict(int))
        for word, sign in [(target, 1), (original, -1)]:
            path = self.paths[word]
            for i, tok in enumerate(path):
                coefficients[path[:i]][tok] += sign
        gradients = {}
        for prefix, counts in coefficients.items():
            counts = {tok: weight for tok, weight in counts.items() if weight}
            if not counts:
                continue
            with torch.enable_grad():
                result = self.engine.run(self.prompt.ids + list(prefix), capture=sites, grad=True, logits_positions=[-1])
                logp = result.logits[0, -1].float().log_softmax(-1)
                objective = sum(weight * logp[tok] for tok, weight in counts.items())
                derivatives = torch.autograd.grad(objective, [result.states[s] for s in sites])
            for site, deriv in zip(sites, derivatives):
                reduced = deriv.detach()[0, positions].float().clone()
                gradients[site] = gradients.get(site, 0) + reduced
            del objective, derivatives, result, logp
        return gradients


def snapshot(engine, prompt, sites):
    run = engine.run(prompt.ids, capture=sites, logits_positions=[-1])
    pos = list(prompt.positions.values())
    return ({s: run.states[s][0, pos].detach().clone() for s in sites},
            run.logits[0, -1].detach().float().clone())


def unit_rows(rows):
    return rows / np.linalg.norm(rows, axis=-1, keepdims=True).clip(1e-8)


class Readers:
    """A frozen cosine k-NN reader and ridge probes see one answer-position vector."""

    def __init__(self, engine):
        sites = [(i, engine.width) for i in range(1, 5)]
        data = defaultdict(list)
        labels, digits = [], []
        for family in families("train"):
            for carry, pair in enumerate(family["sums"]):
                for language in LANGUAGES:
                    prompt = make_prompt(engine, pair, language)
                    run = engine.run(prompt.ids, capture=sites, logits_positions=[-1])
                    for stage, offset in sites:
                        v = run.states[stage, offset][0, -1].detach().float().cpu().numpy()
                        data[stage].append(v.copy())
                    labels.append(carry)
                    digits.append(family["total"] // 10)
                    del run
        self.labels = np.asarray(labels)
        self.vectors, self.weights = {}, {}
        outputs = np.concatenate(((2 * self.labels[:, None] - 1),
                                  np.eye(10)[digits]), axis=1)
        self.bias = outputs.mean(axis=0)
        for stage, vectors in data.items():
            x = unit_rows(np.asarray(vectors, dtype=np.float64))
            self.vectors[stage] = x
            centered = x - x.mean(axis=0)
            dual = np.linalg.solve(centered @ centered.T + 0.05 * np.eye(len(x)),
                                   outputs - self.bias)
            self.weights[stage] = centered.T @ dual

    def read(self, raw, stage):
        x = unit_rows(raw.detach().float().cpu().numpy().astype(np.float64))
        def ridge(train_stage):
            center = self.vectors[train_stage].mean(axis=0)
            return (x - center) @ self.weights[train_stage] + self.bias
        similarity = self.vectors[1] @ x
        nearest = np.argsort(similarity, kind="stable")[-5:]
        fresh = ridge(stage)
        return dict(fixed_knn=int(self.labels[nearest].sum() >= 3),
                    fixed_ridge=int(ridge(1)[0] >= 0), fresh_ridge=int(fresh[0] >= 0),
                    fresh_tens=int(fresh[1:].argmax()))


def measure_family(engine, family, readers):
    idx = family["index"]
    offset = engine.width
    primary_sites = [(stage, offset) for stage in range(1, 5)]
    near_stage = 1 + (idx // 2) % 4
    sites = primary_sites + [(near_stage, offset - 1)] if offset > 1 else primary_sites
    cache = {}
    def acquire(pair, lang):
        key = (tuple(pair), lang)
        if key not in cache:
            p = make_prompt(engine, pair, lang)
            cache[key] = (p, *snapshot(engine, p, sites))
        return cache[key]

    rows, reader_rows, clean_rows = [], [], []
    true_digit = family["total"] // 10
    for carry, pair in enumerate(family["sums"]):
        language = LANGUAGES[(idx + carry) % 2]
        other = LANGUAGES[1 - LANGUAGES.index(language)]
        prompt, clean_states, logits = acquire(pair, language)
        scorer = Scorer(engine, prompt)
        clean = scorer.scores(ALL_WORDS, logits)
        original = WORDS[language][true_digit]
        target_digit = true_digit + 1 - 2 * carry
        target = WORDS[language][target_digit]
        clean_margin = clean["scores"][target] - clean["scores"][original]
        clean_rows.append(dict(carry=carry, language=language, operands=pair,
                               target_digit=target_digit, margin=clean_margin,
                               correct_word_probability=math.exp(clean["scores"][original]),
                               prompt=prompt.text, ids=prompt.ids, positions=prompt.positions,
                               word_token_ids=scorer.paths, **clean))
        for s in primary_sites:
            reader_rows.append(dict(stage=s[0], language=language, carry=carry,
                                    **readers.read(clean_states[s][-1], s[0])))
        roles = list(prompt.positions)
        measured = [(s, "answer position") for s in sites]
        # Rotate operand probes across families; allocation is independent of gradients.
        measured += [(s, roles[(idx // 2 + s[0]) % 2]) for s in primary_sites]
        pos = list(prompt.positions.values())
        gradients = scorer.gradients(target, original, sites, pos)
        words = list(dict.fromkeys([target, original, WORDS[other][target_digit],
                                   WORDS[other][true_digit]]))
        controls = [
            ("opposite/same language", family["sums"][1-carry], language),
            ("opposite/cross language", family["sums"][1-carry], other),
            ("same carry sham", family["sham"][carry], language),
        ]
        for control, donor_pair, donor_lang in controls:
            donor, donor_states, _ = acquire(donor_pair, donor_lang)
            for s, role in measured:
                i = roles.index(role)
                dp, rp = donor.positions[role], prompt.positions[role]
                value = donor_states[s][i]
                delta = value.float() - clean_states[s][i].float()
                predicted = float(torch.dot(gradients[s][i], delta).item())
                def edit(stage, block, state):
                    if (stage, block) != s:
                        return state
                    patched = state.clone()
                    patched[0, rp] = value
                    return patched
                patched_logits = engine.run(prompt.ids, edit_fn=edit, logits_positions=[-1]).logits[0, -1].detach().float()
                result = scorer.scores(words, patched_logits, edit)
                margin = result["scores"][target] - result["scores"][original]
                rows.append(dict(stage=s[0], offset=s[1], role=role, carry=carry,
                                 control=control, language=language, donor_language=donor_lang,
                                 donor_operands=donor_pair, recipient_pos=rp, donor_pos=dp,
                                 recipient_token_id=prompt.ids[rp], donor_token_id=donor.ids[dp],
                                 target_digit=target_digit, target_word=target, clean_word=original,
                                 margin=margin, predicted=predicted, observed=margin-clean_margin,
                                 target_probability=math.exp(result["scores"][target]), **result))
    return dict(family=family, clean=clean_rows, readers=reader_rows, patches=rows)


def bootstrap(records):
    """Resample family means to preserve all within-family dependencies."""
    groups = defaultdict(list)
    for family, value in records:
        groups[family].append(value)
    x = np.array([np.mean(v) for v in groups.values()])
    rng = np.random.default_rng(143)
    samples = x[rng.integers(len(x), size=(2000, len(x)))].mean(axis=1)
    return dict(mean=float(x.mean()), lo=float(np.quantile(samples, .025)),
                hi=float(np.quantile(samples, .975)), families=len(x))


def summarize(results, width):
    groups = defaultdict(list)
    def add(key, value):
        groups[key].append((idx, value))
    shuffled = defaultdict(list)
    for result in results:
        f = result["family"]
        idx = f["index"]
        tens = f["total"] // 10
        for row in result["clean"]:
            lang = row["language"]
            correct_word = WORDS[lang][tens]
            add(("clean", lang, "digit accuracy"), row["digit"] == tens)
            add(("clean", lang, "greedy correct word"), row["first_word"] == correct_word)
            add(("clean", lang, "candidate correct word"), row["best_word"] == correct_word)
            add(("clean", lang, "malformed"), row["malformed"])
            add(("clean", lang, "language unambiguous correct"), row["languages"] == [lang])
            add(("clean", lang, "language ambiguous"), len(row["languages"]) > 1)
            add(("clean", lang, "correct word probability"), row["correct_word_probability"])
        for row in result["readers"]:
            key = (row["language"], row["stage"])
            for name in ("fixed_knn", "fixed_ridge", "fresh_ridge"):
                add(("reader", *key, name), row[name] == row["carry"])
            add(("reader", *key, "fresh_tens"), row["fresh_tens"] == tens)
            shuffled[key].append((idx, row["fixed_knn"], row["carry"]))
        paired = defaultdict(dict)
        for row in result["patches"]:
            key = (row["language"], row["stage"], row["offset"], row["role"])
            base = ("patch", *key, row["control"])
            for metric in ("observed", "predicted", "target_probability"):
                add((*base, metric), row[metric])
            add((*base, "linearization error"), row["predicted"] - row["observed"])
            add((*base, "greedy target digit"), row["digit"] == row["target_digit"])
            add((*base, "greedy target word"), row["first_word"] == row["target_word"])
            add((*base, "language unambiguous correct"), row["languages"] == [row["language"]])
            add((*base, "malformed"), row["malformed"])
            paired[key][row["control"]] = row["observed"]
        for key, scores in paired.items():
            add(("paired", *key, "opposite minus sham"),
                scores["opposite/same language"] - scores["same carry sham"])
            add(("paired", *key, "cross minus same language"),
                scores["opposite/cross language"] - scores["opposite/same language"])
    rng = random.Random(3209)
    for key, rows in shuffled.items():
        preds = [p for _, p, _ in rows]
        rng.shuffle(preds)
        for (idx, _, carry), pred in zip(rows, preds):
            add(("reader", *key, "shuffled vector"), int(pred == carry))
    totals = {r["family"]["index"]: r["family"]["total"] for r in results}
    output = []
    for key, values in groups.items():
        total_interval = bootstrap([(totals[idx], value) for idx, value in values])
        output.append(dict(group=list(key), **bootstrap(values),
                           total_clustered=total_interval))
    return output


def figure(summary, width, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    lookup = {tuple(row["group"]): row["mean"] for row in summary}
    roles = ["left operand", "right operand", "answer position"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 6.5), gridspec_kw={"height_ratios":[3,1]},
                            constrained_layout=True)
    panels = []
    for lang in LANGUAGES:
        for metric in ("predicted", "observed"):
            a = [[lookup.get(("patch", lang, s, width, role, "opposite/cross language",
                              metric), np.nan) for s in range(1, 5)] for role in roles]
            panels.append(np.array(a))
    lim = max(1e-6, float(np.nanmax(np.abs(panels))))
    for col, values in enumerate(panels):
        lang = LANGUAGES[col//2]
        ax = axes[0,col]
        heat = ax.imshow(values, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
        ax.set_title(f"{lang} · {['gradient prediction','actual swap'][col%2]}")
        ax.set_yticks(range(len(roles)), roles if col == 0 else [])
        ax.set_xticks(range(4), [1,2,3,4])
        ax.set_xlabel("Stage")
        ax = axes[1,col]
        for metric, label in (("fixed_ridge", "Stage 1 ridge"), ("fresh_ridge", "Stage-specific")):
            ax.plot(range(1,5), [lookup["reader", lang, s, metric] for s in range(1,5)],
                    marker="o", label=label)
        ax.axhline(.5, color="gray", linestyle=":")
        ax.set_ylim(0,1.04)
        ax.set_xticks(range(1,5))
        if col == 0:
            ax.set_ylabel("Carry accuracy")
            ax.legend(fontsize=7)
    fig.colorbar(heat, ax=axes[0,:], label="Change in full-word log-probability margin")
    fig.suptitle("Carry transfer from an opposite-language donor")
    fig.savefig(out, dpi=140)
    plt.close(fig)


def save_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, allow_nan=False))
    tmp.replace(path)


def run(engine, phase, outdir):
    if phase not in ("smoke", "pilot", "heldout"):
        raise ValueError(phase)
    out = Path(outdir) / f"carry-{engine.name}-{phase}"
    out.mkdir(parents=True, exist_ok=True)
    items = families("heldout" if phase == "heldout" else "pilot")
    if phase == "smoke":
        items = items[:1]
    # Reject stale resumptions when experiment code or the engine has changed.
    import inspect
    fingerprint = hashlib.sha256((inspect.getsource(inspect.getmodule(run)) +
                                  inspect.getsource(type(engine))).encode()).hexdigest()
    meta = dict(phase=phase, model=engine.name, width=engine.width, fingerprint=fingerprint,
                allocated_families=len(items), reader_training_families=SIZES["train"],
                stages=[1,2,3,4], primary_offset=engine.width, nearby_offset=engine.width-1,
                split_units=UNITS, demo_totals_excluded=[32, 44], seeds=SEEDS, prompt=INSTRUCTION,
                primary_physical_blocks_one_based=[engine.width if engine.name == "ouro"
                                                   else stage * engine.width for stage in (1,2,3,4)],
                scoring="full-word log probability, canonical leading space; greedy first word",
                reader="cosine 5-NN and ridge (0.05), labels yes=1, no=0")
    manifest = out / "manifest.json"
    if manifest.exists() and json.loads(manifest.read_text()) != meta:
        raise ValueError(f"Different carry run already exists at {out}")
    save_json(manifest, meta)
    readers = None
    results = []
    for family in items:
        path = out / f"family-{family['index']:03d}.json"
        if path.exists():
            result = json.loads(path.read_text())
        else:
            if readers is None:
                readers = Readers(engine)
            result = measure_family(engine, family, readers)
            save_json(path, result)
            print(f"carry {engine.name} {phase}: {family['index']+1}/{len(items)}", flush=True)
        results.append(result)
    summary = summarize(results, engine.width)
    save_json(out / "summary.json", summary)
    figure(summary, engine.width, out / "carry.png")
    return dict(families=len(results), patches=sum(len(r["patches"]) for r in results),
                directory=out.name)


In [ ]:
%%writefile /kaggle/temp/loop-source/paired.py
"""Controlled comparison on three symbolic tasks. Run with paired.run(output_path).

Every architecture sees the same fixed-length symbolic prompts in the same order
within seed. Stage-end answer supervision is identical. Untied stages start as
exact copies of the first stage and then train independently. The narrow untied
control changes width to approximate the tied parameter count. Findings apply to
these supervised tasks and training budget, independently of the frozen LMs.
"""
import copy
import hashlib
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


STAGES = 4
BLOCKS = 2
WIDTH = 64
HEADS = 4
BATCH = 48
STEPS = 900
SEEDS = (41, 73, 107)
HELDOUT = 128
TASKS = ("carry", "memory", "cue")
NUMBERS = ("zero one two three four five six seven eight nine".split(),
           "zéro un deux trois quatre cinq six sept huit neuf".split())
PLACES = [f"box{i}" for i in range(16)]
FRUITS = "apple pear plum grape mango lemon lime peach".split()
COLORS = "red blue green pink teal gold white black".split()
PEOPLE = "Rae Jo Li Pat Kim Max Sam Lou".split()
PREFIX = "carry ten plus EN FR answer scene put move nobody slot ask Alice Bob R1 R2 world holds uses has".split()
VOCAB = list(dict.fromkeys(PREFIX + list("0123456789") + sum([list(v) for v in NUMBERS], []) +
                           PLACES + FRUITS + COLORS + PEOPLE))
IDS = {token: i for i, token in enumerate(VOCAB)}
SOURCE, QUESTION = 9, 17


def partition(key):
    # Split before drawing question, language, witness, or operand order.
    bucket = int.from_bytes(hashlib.sha256(repr(key).encode()).digest()[:2], "big") % 10
    return "train" if bucket < 6 else "reader" if bucket == 6 else "pilot" if bucket == 7 else "heldout"


def arithmetic_pools():
    pooled_pairs = defaultdict(list)
    for a in range(10, 80):
        for b in range(a, 80):
            total = a + b
            if 30 <= total < 90:
                carry = int(a % 10 + b % 10 >= 10)
                pooled_pairs[total, carry].append((a, b))
    # Stratify assignments within each total and carry class. The raw operand
    # pairs stay disjoint, with enough same-carry shams in evaluation strata.
    schedule = ("train", "heldout", "pilot", "reader", "train", "heldout", "pilot", "reader",
                "train", "train", "heldout", "pilot")
    pools = defaultdict(lambda: defaultdict(list))
    for key, pairs in pooled_pairs.items():
        random.Random(repr(key)).shuffle(pairs)
        for i, pair in enumerate(pairs):
            pools[schedule[i % len(schedule)]][key].append(pair)
    return pools


POOLS = arithmetic_pools()
TOTALS = {split: [t for t in range(30, 90) if min(len(pool[t,c]) for c in (0,1)) >= 2]
          for split, pool in POOLS.items()}


def family(task, split, rng):
    if task == "carry":
        pools = POOLS[split]
        total = rng.choice(TOTALS[split])
        return tuple(rng.sample(pools[total, 0], 2) + rng.sample(pools[total, 1], 2))
    while True:
        if task == "memory":
            scene = tuple(rng.sample(PLACES, 3))
        else:
            scene = (*rng.sample(COLORS, 2), *rng.sample(FRUITS, 2), rng.choice(PEOPLE))
        key = (task, *scene) if task == "memory" else (task, tuple(sorted(zip(scene[:2], scene[2:4]))), scene[4])
        if partition(key) == split:
            return scene


def carry_item(pair, language):
    a, b = pair
    seq = ["carry", *str(a), "plus", *str(b), "ten", ("EN", "FR")[language], "answer"]
    return seq, NUMBERS[language][(a + b) // 10]


def memory_item(scene, register, witness, query):
    actors = ("R1", "R2") if register else ("Alice", "Bob")
    source = ["scene", "put", scene[0], "move", scene[1], actors[witness], "move", scene[2], "nobody"]
    question = ["ask", (*actors, "world")[query], "answer"]
    answer = scene[2] if query == 2 else scene[int(query == witness)]
    return source + ["slot"] * 8 + question, answer


def cue_item(scene, selected, variant="two"):
    c0, c1, f0, f1, person = scene
    desc = [c0, "holds", f0, c1, "holds", f1]
    clue = [person, "uses", (c0, c1)[selected]]
    record = (clue + desc) if variant == "first" else (desc + clue)
    prefix = [] if variant == "one" else ["slot"] * 9 if variant == "filler" else record
    return prefix + record + [person, "has", "answer"], (f0, f1)[selected]


def cue_decoy_item(scene, selected, decoy_choice, relevant_first):
    c0, c1, f0, f1, person = scene
    decoy = PEOPLE[(PEOPLE.index(person) + 1) % len(PEOPLE)]
    target = [person, "uses", (c0, c1)[selected]]
    irrelevant = [decoy, "uses", (c0, c1)[decoy_choice]]
    descriptions = [c0, "holds", f0, c1, "holds", f1]
    record = descriptions + (target + irrelevant if relevant_first else irrelevant + target)
    return record * 2 + [person, "has", "answer"], (f0, f1)[selected]


def encode(items, device):
    tokens, labels = zip(*items)
    return (torch.tensor([[IDS[word] for word in seq] for seq in tokens], device=device),
            torch.tensor([IDS[word] for word in labels], device=device))


class Block(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm1 = nn.LayerNorm(width)
        self.norm2 = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.project = nn.Linear(width, width)
        self.mlp = nn.Sequential(nn.Linear(width, 3 * width), nn.GELU(), nn.Linear(3 * width, width))

    def forward(self, x, allowed):
        batch, length, width = x.shape
        q, k, v = self.qkv(self.norm1(x)).view(batch, length, 3, HEADS, width // HEADS).permute(2, 0, 3, 1, 4)
        attention = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
        x = x + self.project(attention.transpose(1, 2).reshape_as(x))
        return x + self.mlp(self.norm2(x))


class TinyLM(nn.Module):
    def __init__(self, width, tied):
        super().__init__()
        self.embedding = nn.Embedding(len(VOCAB), width)
        self.position = nn.Embedding(32, width)
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, len(VOCAB))
        first = nn.ModuleList(Block(width) for _ in range(BLOCKS))
        self.stacks = nn.ModuleList([first] + ([] if tied else [copy.deepcopy(first) for _ in range(3)]))
        self.register_buffer("allowed", torch.ones(32, 32, dtype=torch.bool).tril(), persistent=False)

    def forward(self, tokens, mode="causal", cuts=(), edit=None, return_states=False,
                offset=0, cut_rows=slice(9, 15), cut_cols=slice(6, 9)):
        length = tokens.shape[1]
        x = self.embedding(tokens) + self.position(torch.arange(offset, offset + length, device=tokens.device))
        causal = self.allowed[:length, :length]
        logits, states = [], []
        for stage in range(STAGES):
            for block, layer in enumerate(self.stacks[stage % len(self.stacks)]):
                site = stage * BLOCKS + block  # zero-based executed block
                mask = causal
                if mode in ("memory", "sealed"):
                    mask = causal.clone()
                    if stage < 2:
                        mask[QUESTION:, :QUESTION] = False
                    if stage >= 2 or mode == "sealed":
                        mask[SOURCE:, :SOURCE] = False
                elif mode == "isolated":
                    mask = causal.clone()
                    mask[9:, :9] = False
                if site in cuts:
                    mask = mask.clone()
                    mask[cut_rows, cut_cols] = False
                x = layer(x, mask)
                if edit is not None:
                    x = edit(site, x)
                if return_states:
                    states.append(x)
            logits.append(self.head(self.norm(x[:, -1])))
        return torch.stack(logits), states


def clean_metrics(logits, answers, allowed):
    probability = logits.softmax(-1)
    guess = logits.argmax(-1)
    return {"accuracy": (guess == answers).float(), "answer_mass": probability.gather(1, answers[:, None])[:, 0],
            "candidate_mass": probability[:, allowed].sum(1), "malformed": (~torch.isin(guess, allowed)).float()}


def training_item(task, scene, rng):
    if task == "carry":
        return carry_item(rng.choice(scene), rng.randrange(2))
    if task == "memory":
        return memory_item(scene, rng.randrange(2), rng.randrange(2), rng.randrange(3))
    return cue_item(scene, rng.randrange(2))


def train(model, device, seed, steps, batch):
    rng = random.Random(200000 + seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.0015, weight_decay=.01)
    token_count, digest, loss = 0, hashlib.sha256(), 0.
    model.train()
    for i in range(steps):
        task = TASKS[i % len(TASKS)]
        # Keep token lengths uniform inside each minibatch. Expose both clue formats.
        if task == "cue" and (i // 3) % 2:
            items = [cue_decoy_item(family(task, "train", rng), rng.randrange(2), rng.randrange(2),
                                    rng.randrange(2)) for _ in range(batch)]
        else:
            items = [training_item(task, family(task, "train", rng), rng) for _ in range(batch)]
        x, y = encode(items, "cpu")
        digest.update(x.numpy().tobytes())
        digest.update(y.numpy().tobytes())
        token_count += x.numel()
        x, y = x.to(device), y.to(device)
        # Both architectures learn the same readout at every answerable stage.
        logits, _ = model(x, mode="memory" if task == "memory" else "causal")
        stages = logits[2:] if task == "memory" else logits
        loss = F.cross_entropy(stages.flatten(0, 1), y.repeat(stages.shape[0]))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        for group in optimizer.param_groups:
            group["lr"] = .0002 + .0013 * (1 + math.cos(math.pi * i / steps)) / 2
        optimizer.step()
        if (i + 1) % 300 == 0:
            print(f"Controlled training seed {seed}, step {i+1}/{steps}, loss {loss.item():.3f}", flush=True)
    model.eval()
    return {"train_input_tokens": token_count, "stream_sha256": digest.hexdigest(),
            "last_minibatch_loss": loss.detach().item(), "parameters": sum(p.numel() for p in model.parameters())}


def dataset(task, split, size):
    rng = random.Random(7812 if split == "pilot" else 39452)
    scenes = {}
    for attempt in range(size * 1000):
        if len(scenes) == size:
            break
        scene = family(task, split, rng)
        # A repeated carry pair in reverse sham order does not create a fresh family.
        key = tuple(sorted(scene)) if task == "carry" else scene
        scenes[key] = scene
    if len(scenes) != size:
        raise RuntimeError(f"Found {len(scenes)} of {size} unique {task} {split} families")
    return list(scenes.values())


def frozen_reader(states, labels):
    x = torch.cat([states, torch.ones_like(states[:, :1])], dim=1).double()
    reg = .1 * torch.eye(x.shape[1], device=x.device, dtype=x.dtype)
    return torch.linalg.solve(x.T @ x + reg, x.T @ (labels.double() * 2 - 1))


def read(reader, states):
    return states.double() @ reader[:-1] + reader[-1]


@torch.no_grad()
def fit_readers(model, device):
    scenes = dataset("carry", "reader", 32)
    inputs = [(carry_item(scene[carry * 2], lang), carry) for scene in scenes for carry in range(2) for lang in range(2)]
    codes = [[] for _ in range(STAGES)]
    for start in range(0, len(inputs), 32):
        x, _ = encode([item for item, _ in inputs[start:start+32]], device)
        _, states = model(x, return_states=True)
        for stage in range(STAGES):
            codes[stage].append(states[stage * BLOCKS + BLOCKS - 1][:, -1])
    labels = torch.tensor([c for _, c in inputs], device=device)
    return [frozen_reader(torch.cat(xs), labels) for xs in codes]


def carry_eval(model, scenes, device, readers):
    metrics = defaultdict(list)
    candidates = torch.tensor([IDS[n] for lang in NUMBERS for n in lang], device=device).unique()
    reader_codes = defaultdict(list)
    with torch.enable_grad():
        for lang in (0, 1):
            for carry in (0, 1):
                x, answer = encode([carry_item(s[2*carry], lang) for s in scenes], device)
                counter = torch.tensor([IDS[NUMBERS[lang][sum(s[0]) // 10 + 1 - 2 * carry]] for s in scenes], device=device)
                logits, states = model(x, return_states=True)
                margin = (logits[-1].gather(1, counter[:, None]) - logits[-1].gather(1, answer[:, None]))[:, 0]
                sites = [1, 3, 5, 7]
                grad = torch.autograd.grad(margin.sum(), [states[i] for i in sites])
                with torch.no_grad():
                    for k, v in clean_metrics(logits[-1], answer, candidates).items():
                        metrics["clean_" + k].append(v)
                    for stage, site in enumerate(sites):
                        code = states[site][:, -1]
                        reader_codes[stage].append(code.detach())
                        metrics[f"s{stage+1}_reader_fixed"].append((read(readers[0], code).gt(0) == bool(carry)).float())
                        metrics[f"s{stage+1}_reader_local"].append((read(readers[stage], code).gt(0) == bool(carry)).float())
                    for control, index, donor_lang in (("swap", 2*(1-carry), lang),
                                                        ("cross_language", 2*(1-carry), 1-lang),
                                                        ("sham", 2*carry+1, lang)):
                        donor, _ = encode([carry_item(s[index], donor_lang) for s in scenes], device)
                        _, donor_states = model(donor, return_states=True)
                        for stage, site in enumerate(sites):
                            value = donor_states[site][:, -1]
                            predicted = (grad[stage][:, -1] * (value - states[site][:, -1])).sum(-1)
                            def swap(idx, hidden):
                                if idx != site:
                                    return hidden
                                z = hidden.clone()
                                z[:, -1] = value
                                return z
                            patched = model(x, edit=swap)[0][-1]
                            actual = (patched.gather(1, counter[:, None]) - patched.gather(1, answer[:, None]))[:, 0] - margin
                            stem = f"s{stage+1}_{control}"
                            metrics[stem + "_observed"].append(actual)
                            if control != "sham":
                                direction = "remove" if carry else "add"
                                lang_tag = "FR" if lang else "EN"
                                metrics[f"{stem}_{lang_tag}_{direction}"].append(actual)
                            metrics[stem + "_predicted"].append(predicted)
                            lang_ids = torch.tensor([IDS[n] for n in NUMBERS[lang]], device=device)
                            metrics[stem + "_recipient_language"].append(torch.isin(patched.argmax(-1), lang_ids).float())
    for stage, codes in reader_codes.items():
        population = torch.cat(codes)
        generator = torch.Generator().manual_seed(521 + stage)
        order = torch.randperm(len(population), generator=generator).to(device)
        labels = torch.tensor([0, 1, 0, 1], device=device).repeat_interleave(len(scenes))
        hits = read(readers[0], population[order]).gt(0).eq(labels).float()
        metrics[f"s{stage+1}_reader_shuffled"].append(hits.reshape(4, -1).mean(0))
    return {k: torch.stack(v).detach().mean(0) for k, v in metrics.items()}


@torch.no_grad()
def memory_eval(model, scenes, device):
    metrics = defaultdict(list)
    candidates = torch.tensor([IDS[n] for n in PLACES], device=device)
    for register, group in ((0, "observer"), (1, "register")):
        for witness in (0, 1):
            common, predictions = None, []
            for query in range(3):
                x, answer = encode([memory_item(s, register, witness, query) for s in scenes], device)
                original, states = model(x, mode="memory", return_states=True)
                memory = states[3][:, SOURCE:QUESTION]
                if common is None:
                    common = memory
                else:
                    torch.testing.assert_close(memory, common, atol=1e-5, rtol=1e-5)
                def replace_memory(value):
                    def edit(site, h):
                        if site != 3:
                            return h
                        z = h.clone()
                        z[:, SOURCE:QUESTION] = value
                        return z
                    return edit
                logits = model(x, mode="memory", edit=replace_memory(common))[0]
                torch.testing.assert_close(original, logits, atol=1e-5, rtol=1e-5)
                shuffled = model(x, mode="memory", edit=replace_memory(common.roll(1, 0)))[0][-1]
                direct = model(x)[0][-1]
                sealed = model(x, mode="sealed")[0][-1]
                changed = x.clone()
                changed[:, :SOURCE] = x.roll(1, 0)[:, :SOURCE]
                torch.testing.assert_close(sealed, model(changed, mode="sealed")[0][-1], atol=1e-5, rtol=1e-5)
                for name, head in (("direct", direct), ("shuffled", shuffled), ("s3", logits[2]), ("s4", logits[3])):
                    for k, v in clean_metrics(head, answer, candidates).items():
                        metrics[f"{group}_{name}_{k}"].append(v)
                if query < 2:
                    world = torch.tensor([IDS[s[2]] for s in scenes], device=device)
                    metrics[group+"_late_world_drift"].append((logits[3].argmax(-1) == world).float() -
                                                               (logits[2].argmax(-1) == world).float())
                predictions.append(logits[3].argmax(-1) == answer)
            metrics[group+"_all_three"].append(torch.stack(predictions).all(0).float())
    return {k: torch.stack(v).mean(0) for k, v in metrics.items()}


@torch.no_grad()
def cue_eval(model, scenes, device):
    metrics = defaultdict(list)
    candidates = torch.tensor([IDS[n] for n in FRUITS], device=device)
    for selected in (0, 1):
        x, y = encode([cue_item(s, selected) for s in scenes], device)
        other = torch.tensor([IDS[s[3 - selected]] for s in scenes], device=device)
        isolated = model(x, mode="isolated")[0]
        torch.testing.assert_close(isolated, model(x[:, 9:], offset=9)[0], atol=2e-5, rtol=2e-5)
        for variant in ("two", "one", "filler", "first"):
            inp, _ = encode([cue_item(s, selected, variant) for s in scenes], device)
            for k, v in clean_metrics(model(inp)[0][-1], y, candidates).items():
                metrics[f"{variant}_{k}"].append(v)
        for control, early, late, rows, cols in (("matched", 2, 4, slice(9, 15), slice(6, 9)),
                                                  ("offset", 2, 5, slice(9, 15), slice(6, 9))):
            margins = []
            for label, cuts in (("open", ()), ("early", (early,)), ("late", (late,)), ("both", (early, late))):
                logits = model(x, cuts=cuts, cut_rows=rows, cut_cols=cols)[0][-1]
                margin = (logits.gather(1, y[:, None]) - logits.gather(1, other[:, None]))[:, 0]
                metrics[f"{control}_{label}_margin"].append(margin)
                margins.append(margin)
            metrics[control+"_interaction"].append(margins[1] - margins[3] - margins[0] + margins[2])
        # In the two-person prompt, compare equal-area cuts and counterbalance
        # both clues' locations. Independent decoy choices cancel incidental agreement.
        for decoy_choice in (0, 1):
            for relevant_first in (False, True):
                inputs, _ = encode([cue_decoy_item(s, selected, decoy_choice, relevant_first) for s in scenes], device)
                intact = model(inputs)[0][-1]
                metrics["two_person_accuracy"].append(intact.argmax(1).eq(y).float())
                for relevant in (False, True):
                    start = 6 if relevant == relevant_first else 9
                    prefix = "target" if relevant else "irrelevant"
                    margins = []
                    for cuts in ((), (2,), (4,), (2, 4), tuple(range(8))):
                        logits = (model(inputs, cuts=cuts, cut_rows=slice(12, 18),
                                        cut_cols=slice(start, start+3))[0][-1] if cuts else intact)
                        margins.append((logits.gather(1, y[:, None]) - logits.gather(1, other[:, None]))[:, 0])
                    metrics[prefix+"_interaction"].append(margins[1] - margins[3] - margins[0] + margins[2])
                    metrics[prefix+"_all_layer_cost"].append(margins[0] - margins[4])
    return {k: torch.stack(v).mean(0) for k, v in metrics.items()}


def evaluate(model, device, split, n, readers):
    rows = {}
    for task, evaluate_task in zip(TASKS, (carry_eval, memory_eval, cue_eval)):
        scenes = dataset(task, split, n)
        task_rows = []
        for start in range(0, n, 24):
            chunk = scenes[start:start + 24]
            if len(chunk) < 2 and task == "memory":
                raise ValueError("A shuffled-memory evaluation batch needs at least two scenes")
            values = evaluate_task(model, chunk, device, readers) if task == "carry" else evaluate_task(model, chunk, device)
            values = {key: vector.cpu().tolist() for key, vector in values.items()}
            for i, scene in enumerate(chunk):
                if task == "carry":
                    cluster = str(sum(scene[0]))
                elif task == "cue":
                    cluster = repr((sorted(zip(scene[:2], scene[2:4])), scene[4]))
                else:
                    cluster = repr(scene)
                task_rows.append({"family": scene, "cluster": cluster, "metrics": {k: v[i] for k, v in values.items()}})
        rows[task] = task_rows
    return rows


def write_json(path, data):
    tmp = path.with_suffix(".pending")
    tmp.write_text(json.dumps(data, separators=(",", ":"), allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def count(tied, width):
    return sum(p.numel() for p in TinyLM(width, tied).parameters())


def matched_width(width):
    reference = count(True, width)
    return min(range(HEADS, width + 1, HEADS), key=lambda d: abs(count(False, d) - reference))


def bootstrap(values, clusters, rng):
    # Resample complete clusters. Carry families sharing a total stay together.
    keys, ix = np.unique(clusters, return_inverse=True)
    totals = np.bincount(ix, weights=values)
    sizes = np.bincount(ix)
    draws = rng.integers(len(keys), size=(2000, len(keys)))
    samples = totals[draws].sum(1) / sizes[draws].sum(1)
    return {"mean": float(np.mean(values)), "low": float(np.quantile(samples, .025)),
            "high": float(np.quantile(samples, .975)), "clusters": len(keys)}


def summarize(results, seeds):
    rng = np.random.default_rng(924051)
    comparisons, individual = {}, {}
    for task in TASKS:
        sample = results[seeds[0], "tied"]["heldout"][task]
        clusters = [r["cluster"] for r in sample]
        for key in sample[0]["metrics"]:
            by_arch = {}
            for name in ("tied", "untied", "untied_budget"):
                arr = np.stack([[r["metrics"][key] for r in results[seed, name]["heldout"][task]] for seed in seeds])
                by_arch[name] = arr
                individual.setdefault(task, {}).setdefault(name, {})[key] = {
                    "mean": float(arr.mean()), "by_seed": arr.mean(1).tolist()}
            for name in ("untied", "untied_budget"):
                delta = by_arch["tied"] - by_arch[name]
                stats = bootstrap(delta.mean(0), clusters, rng)
                stats["by_seed"] = delta.mean(1).tolist()
                comparisons.setdefault(task, {}).setdefault("tied_minus_" + name, {})[key] = stats
    return individual, comparisons


def _study(outdir, device, steps=STEPS, batch=BATCH, width=WIDTH, seeds=SEEDS, n=HELDOUT):
    directory = Path(outdir) / "paired"
    directory.mkdir(parents=True, exist_ok=True)
    small = matched_width(width)
    archs = {"tied": (width, True), "untied": (width, False), "untied_budget": (small, False)}
    settings = {"seeds": list(seeds), "steps": steps, "batch": batch, "width": width, "n_heldout": n,
                "n_pilot": 24, "blocks_per_stage": BLOCKS, "stages": STAGES, "narrow_width": small,
                "heads": HEADS, "optimizer": "AdamW", "readout_stages": {"memory": [3, 4], "carry": [1, 2, 3, 4],
                                                                        "cue": [1, 2, 3, 4]},
                "vocabulary": VOCAB, "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    results = {}
    for seed in seeds:
        # Functional-equivalence check using complete outputs at every stage.
        # Perform it in FP32 before gradient updates.
        with torch.no_grad():
            probe, _ = encode([cue_item(dataset("cue", "pilot", 1)[0], 0)], device)
            torch.manual_seed(seed)
            ref = TinyLM(width, True).to(device)(probe)[0]
            torch.manual_seed(seed)
            other = TinyLM(width, False).to(device)(probe)[0]
            torch.testing.assert_close(ref, other, atol=0, rtol=0)
        for name, (d, tied) in archs.items():
            path = directory / f"{seed}_{name}.json"
            previous = json.loads(path.read_text()) if path.is_file() else {}
            if previous.get("config") == settings and previous.get("complete") is True:
                results[seed, name] = previous
                continue
            torch.manual_seed(seed)
            model = TinyLM(d, tied).to(device)
            started = time.perf_counter()
            info = train(model, device, seed, steps, batch)
            for param in model.parameters():
                param.requires_grad_(False)
            # Input embeddings participate in activation attribution, never optimization here.
            model.embedding.weight.requires_grad_(True)
            readers = fit_readers(model, device)
            output = {"config": settings, "architecture": name, "seed": seed, "training": info,
                      "pilot": evaluate(model, device, "pilot", 24, readers),
                      "heldout": evaluate(model, device, "heldout", n, readers),
                      "elapsed_seconds": time.perf_counter() - started, "complete": True}
            write_json(path, output)
            results[seed, name] = output
            print(f"Controlled {name}, seed {seed} saved ({output['elapsed_seconds']:.1f}s)", flush=True)
            del model
        streams = {results[seed, name]["training"]["stream_sha256"] for name in archs}
        if len(streams) != 1:
            raise RuntimeError("Training streams differed between architectures")
    individual, comparisons = summarize(results, seeds)
    summary = {"all_runs_complete": True, "config": settings, "means": individual, "contrasts": comparisons,
               "scope": "Symbolic task-trained models. Intervals resample held-out scene families (sums grouped by total), conditional on these training seeds. Per-seed means are included. Carry swaps target the final prompt token. Identical French/English word 'six' is compatible with either language. The cue alternative offset was fixed before testing, without matching individual effect magnitudes. The narrow parameter-count control changes width and forward compute.",
               "device": str(device), "torch_version": torch.__version__,
               "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None}
    write_json(directory / "summary.json", summary)
    return summary


def run(outdir):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle GPU before running the controlled training experiment")
    return _study(outdir, torch.device("cuda"))


In [ ]:
%%writefile /kaggle/temp/loop-source/verify_results.py
"""Verify downloaded Kaggle outputs against the experiment code and raw observations."""
import argparse
from collections import Counter, defaultdict
import hashlib
import itertools
import json
import math
from pathlib import Path
import sys


ENGINES = ("ouro", "qwen")
TASKS = ("carry", "memory", "cue")
PHASES = ("pilot", "heldout")


class AuditError(ValueError):
    pass


def check(condition, message):
    if not condition:
        raise AuditError(message)


def finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def read_json(path):
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)


def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def family_records(families):
    entries = [json.dumps(f, sort_keys=True, separators=(",", ":")) for f in families]
    check(len(set(entries)) == len(entries), "Duplicate scene families")
    digest = hashlib.sha256("\n".join(entries).encode()).hexdigest()
    return {"n_families": len(entries), "family_data_sha256": digest, "_families": set(entries)}


def check_receipt(receipt, source):
    check(receipt["status"] == "complete", "Receipt did not report completion")
    check(receipt["platform"] == "kaggle", "Receipt does not identify a Kaggle run")
    gpu = receipt["gpu"]
    check(isinstance(gpu, str) and gpu.strip() and gpu.lower() != "cpu",
          "Receipt does not identify GPU hardware")
    for key in ("elapsed_seconds", "peak_allocated_bytes", "peak_reserved_bytes"):
        check(finite(receipt[key]) and receipt[key] > 0, f"Invalid hardware measurement: {key}")
    check(receipt["peak_reserved_bytes"] >= receipt["peak_allocated_bytes"],
          "GPU reservation is smaller than measured allocation")
    hashes = receipt["code_sha256"]
    check(isinstance(hashes, dict) and bool(hashes), "No source hashes in receipt")
    for name, expected in hashes.items():
        check(Path(name).name == name and name not in {".", ".."}, "Invalid source filename")
        check(file_hash(source / name) == expected, f"Source differs from Kaggle run: {name}")
    return {"gpu": gpu, "elapsed_seconds": receipt["elapsed_seconds"],
            "peak_allocated_bytes": receipt["peak_allocated_bytes"],
            "peak_reserved_bytes": receipt["peak_reserved_bytes"]}


def check_native(root, model, receipt):
    diag = read_json(root / f"{model}-native-validation.json")
    revision, width = {
        "ouro": ("574fa66cb8bf5abdc979642d01cf2b79b16bfab1", 24),
        "qwen": ("8faed761d45a263340a0528343f099c05c9a4323", 7),
    }[model]
    check(diag["checkpoint"][1] == revision and diag["blocks_per_stage"] == width
          and diag["executed_depth"] == 4*width, "Wrong checkpoint or executed depth")
    attention = "eager" if model == "ouro" else "sdpa_math"
    check(diag["attention"] == attention and diag["dtype"] == "torch.float16" and diag["use_cache"] is False,
          "Native equivalence test used a different model configuration")
    check(diag["gpu"] == receipt["gpu"] and diag["code_sha256"]["model_engine.py"] ==
          receipt["code_sha256"]["model_engine.py"], "Native equivalence and measurement provenance disagree")
    errors = diag["stage_native_max_abs_errors"]
    check(len(errors) == 4 and all(finite(x) and x >= 0 for x in errors), "Missing stage parity evidence")
    for key in ("final_native_max_abs_error", "stage_four_head_max_abs_error", "sham_final_max_abs_error"):
        check(finite(diag[key]) and diag[key] >= 0, f"Invalid native parity evidence: {key}")


def verify_carry(root, model, phase):
    directory = root / f"carry-{model}-{phase}"
    manifest = read_json(directory / "manifest.json")
    summary = {tuple(r["group"]): r for r in read_json(directory / "summary.json")}
    n = 24 if phase == "pilot" else 100
    check((manifest["model"], manifest["phase"], manifest["allocated_families"]) == (model, phase, n),
          "Carry manifest does not match requested split")
    groups = {"opposite/same language", "opposite/cross language", "same carry sham"}
    check(len(list(directory.glob("family-*.json"))) == n, "Wrong number of carry family files")
    languages = ("English", "French")
    correct = Counter()
    families = []
    for index in range(n):
        row = read_json(directory / f"family-{index:03d}.json")
        family = row["family"]
        families.append({k: family[k] for k in ("sham", "sums", "total")})
        check(family["index"] == index and family["total"] % 10 in manifest["split_units"][phase],
              "Carry family index or split mismatch")
        for group in ("sums", "sham"):
            for carry, operands in enumerate(family[group]):
                a, b = operands
                check(a + b == family["total"] and carry == int(a % 10 + b % 10 >= 10),
                      "Invalid arithmetic carry or matched total")
        check(len(row["clean"]) == 2 and {r["language"] for r in row["clean"]} == set(languages),
              "Missing bilingual clean scores")
        clean = {r["carry"]: r for r in row["clean"]}
        for r in clean.values():
            check(0 <= r["correct_word_probability"] <= 1 and finite(r["margin"]), "Invalid clean carry scores")
            correct[r["language"]] += r["digit"] == family["total"] // 10
        reader_counts = Counter((r["carry"], r["stage"]) for r in row["readers"])
        check(reader_counts == Counter(itertools.product(range(2), range(1,5))), "Missing carry readout")
        by_site = defaultdict(set)
        for patch in row["patches"]:
            c, stage, offset, role = (patch[k] for k in ("carry", "stage", "offset", "role"))
            observed = patch["margin"] - clean[c]["margin"]
            check(finite(patch["predicted"]) and math.isclose(observed, patch["observed"], abs_tol=1e-6),
                  "Carry patch margin is nonfinite or disagrees with its clean baseline")
            check(patch["target_digit"] == family["total"] // 10 + 1 - 2*c, "Wrong carry direction")
            if stage == 4 and offset == manifest["width"] and role != "answer position":
                check(abs(patch["predicted"]) < 1e-5 and abs(patch["observed"]) < 1e-5,
                      "A final-block non-answer position influenced the output")
            by_site[c, stage, offset, role].add(patch["control"])
        check(len(row["patches"]) == 54 and len(by_site) == 18 and all(v == groups for v in by_site.values()),
              "Missing or duplicate carry control, site, or direction")
        for c, stage in itertools.product(range(2), range(1,5)):
            check((c, stage, manifest["width"], "answer position") in by_site, "Missing end-of-stage carry patch")
    measures = {}
    for lang in languages:
        estimate = summary["clean", lang, "digit accuracy"]
        check(estimate["families"] == n and math.isclose(estimate["mean"], correct[lang]/n, abs_tol=1e-8),
              "Carry aggregate disagrees with raw responses")
        for metric in ("digit accuracy", "greedy correct word", "malformed", "correct word probability"):
            measures[f"{lang} {metric}"] = summary["clean", lang, metric]["mean"]
    return {**family_records(families), "measurements": measures}


def verify_memory(root, model, phase):
    directory = root / "memory" / model / phase
    summary = read_json(directory / "summary.json")
    first, stop = (0, 24) if phase == "pilot" else (24, 144)
    n = stop-first
    check((summary["checkpoint"], summary["phase"], summary["sizes"][phase]) == (model, phase, n),
          "Memory summary or split mismatch")
    check(len(list(directory.glob("scene-*.json"))) == n, "Wrong number of memory scenes")
    domains = ("people", "registers")
    roles = ("first", "second", "world")
    required = {(d, toggle, condition, role, stage)
                for d, toggle, role, stage in itertools.product(domains, (False, True), roles, (3,4))
                for condition in ("direct", "memory", "shuffled") if not (toggle and condition == "shuffled")}
    direct, through_memory = Counter(), Counter()
    families = []
    for i in range(first, stop):
        scene = read_json(directory / f"scene-{i:03d}.json")
        if i == first:
            for d in domains:
                for k in ("sham_max_abs", "source_closed_max_abs", "memory_independence_max_abs",
                          "fixed_memory_max_abs"):
                    diff = scene["checks"][d][k]
                    check(finite(diff) and diff >= 0, f"Missing source-closure control: {k}")
        rows = scene["rows"]
        families.append([rows[0]["boxes"], rows[0]["witness"]])
        observed = {tuple(r[k] for k in ("domain", "toggle", "condition", "role", "stage")) for r in rows}
        check(len(rows) == len(required) and observed == required, "Memory scene missing a condition or question")
        for r in rows:
            check(r["family"] == i and r["target"] in r["boxes"] and r["winner"] in r["boxes"],
                  "Invalid family or location in memory result")
            if not r["toggle"] and r["stage"] == 4:
                if r["condition"] == "direct":
                    direct[r["domain"]] += r["target"] == r["winner"]
                if r["condition"] == "memory":
                    through_memory[r["domain"]] += r["target"] == r["winner"]
            check(0 <= r["correct_mass"] <= r["answer_mass"] <= 1.00001, "Invalid box answer probabilities")
    output = {}
    for d in domains:
        for label, counts in (("direct", direct), ("memory", through_memory)):
            key = f"{d}/one/{label}/stage4"
            estimate = summary["metrics"][key]["native_mean_accuracy"]
            check(estimate["families"] == n and math.isclose(counts[d]/(3*n), estimate["mean"], abs_tol=1e-8),
                  "Memory accuracy disagrees with the raw answers")
            output[key] = estimate["mean"]
        output[f"{d}/late_world_intrusion"] = summary["paired_contrasts"][f"{d}/winner"]["late_world_intrusion_change"]
    gate = summary["room_gate"]["passed"]
    if gate:
        check(len(read_json(directory / "room-rows.json")) == 4*n, "Missing gated two-room measurements")
    output["two_room_gate_passed"] = gate
    return {**family_records(families), "measurements": output}


def verify_cue(root, model, phase):
    folder = root / "cue"
    raw = read_json(folder / f"{model}_{phase}_rows.json")
    summary = read_json(folder / f"{model}_{phase}_summary.json")
    sites = read_json(folder / f"{model}_frozen_sites.json")
    rows = raw["rows"]
    first, stop = (0, 24) if phase == "pilot" else (24, 124)
    check([r["scene"]["number"] for r in rows] == list(range(first, stop)),
          "Repeated-clue rows have missing or duplicate scene families")
    check((summary["model"], summary["phase"], summary["n_families"]) ==
          (model, phase, len(rows)) and summary["sites"] == sites, "Clue summary or site selection mismatch")
    check(summary["isolation"] == raw["isolation"], "Clue isolation results disagree")
    for measure, limit in (("isolated_max_abs", "isolated_max_limit"), ("isolated_rms", "isolated_rms_limit"),
                           ("altered_prefix_max_abs", "altered_prefix_limit")):
        iso = raw["isolation"]
        check(finite(iso[measure]) and 0 <= iso[measure] <= iso[limit], f"Cross-copy isolation failed: {measure}")
    neighbor = sites["late_neighbor"]
    required = {"intact", "target_all_layers", "reversed_assignment", "one_copy", "filler", "clue_first",
                f"neighbor_{neighbor}_late", f"neighbor_{neighbor}_both"}
    required.update(f"{region}_{cut}" for region in ("target", "sham") for cut in ("early", "late", "both"))
    measured = defaultdict(list)
    for row in rows:
        scores = row["scores"]
        check(required <= scores.keys(), "Clue row missing a control or cut")
        m = {}
        for name, s in scores.items():
            check(finite(s["margin"]), f"Nonfinite clue margin: {name}")
            probs = [s[k] for k in ("correct_probability", "other_probability")]
            check(all(finite(p) and 0 <= p <= 1 for p in probs) and sum(probs) <= 1.00001,
                  f"Invalid candidate probability mass: {name}")
            m[name] = s["margin"]
        measured["target_interaction"].append(m["target_early"] - m["target_both"] - m["intact"] + m["target_late"])
        measured["sham_interaction"].append(m["sham_early"] - m["sham_both"] - m["intact"] + m["sham_late"])
        measured["nearby_interaction"].append(m["target_early"] - m[f"neighbor_{neighbor}_both"]
                                             - m["intact"] + m[f"neighbor_{neighbor}_late"])
        measured["duplication_gain"].append(m["intact"] - m["one_copy"])
        measured["all_layers_cost"].append(m["intact"] - m["target_all_layers"])
        measured["intact.candidate_accuracy"].append(m["intact"] > 0)
    def geometry(row, key):
        return tuple(row["copy1"][key]) + tuple(row["copy2_queries"])
    check(sorted(geometry(r, "target") for r in rows) == sorted(geometry(r, "sham") for r in rows),
          "Clue and sham mask geometry differs across families")
    estimates = {}
    for key, sample in measured.items():
        estimate = summary["estimates"][key]
        check(all(finite(estimate[k]) for k in ("mean", "lo", "hi")) and estimate["lo"] <= estimate["hi"],
              f"Malformed clue interval: {key}")
        check(math.isclose(estimate["mean"], sum(sample) / len(sample), abs_tol=1e-8, rel_tol=1e-8),
              f"Clue aggregate differs from raw scores: {key}")
        estimates[key] = estimate
    families = [{k: v for k, v in r["scene"].items() if k != "number"} for r in rows]
    return {**family_records(families), "measurements": estimates}


def verify_paired(root, source):
    directory = root / "paired"
    summary = read_json(directory / "summary.json")
    cfg = summary["config"]
    seeds, names = (41, 73, 107), ("tied", "untied", "untied_budget")
    check(summary["all_runs_complete"] is True and summary["device"].startswith("cuda") and summary["gpu"],
          "Missing completed GPU training runs")
    check(tuple(cfg["seeds"]) == seeds and cfg["steps"] == 900 and cfg["n_pilot"] == 24
          and cfg["n_heldout"] == 128 and (cfg["stages"], cfg["blocks_per_stage"]) == (4, 2)
          and cfg["source_sha256"] == file_hash(source / "paired.py"), "Wrong matched-training design or source")
    metrics = {"carry": "s4_cross_language_observed", "memory": "observer_late_world_drift",
               "cue": "matched_interaction"}
    frames, parameters, samples = {}, {}, defaultdict(list)
    for seed in seeds:
        streams = set()
        for name in names:
            result = read_json(directory / f"{seed}_{name}.json")
            check(result["config"] == cfg and result["complete"] is True and
                  (result["architecture"], result["seed"]) == (name, seed), "Paired fit metadata mismatch")
            tr = result["training"]
            streams.add((tr["stream_sha256"], tr["train_input_tokens"]))
            check(tr["train_input_tokens"] > 0 and finite(tr["last_minibatch_loss"]), "Missing training evidence")
            parameters[name] = tr["parameters"]
            for task in TASKS:
                for phase, n in (("pilot", 24), ("heldout", 128)):
                    rows = result[phase][task]
                    scene_ids = [json.dumps(r["family"], separators=(",", ":")) for r in rows]
                    check(len(rows) == n and len(set(scene_ids)) == n, "Missing or duplicate paired families")
                    check(scene_ids == frames.setdefault((phase, task), scene_ids),
                          "Evaluation families differ between compared fits")
                    value = [r["metrics"][metrics[task]] for r in rows]
                    check(all(finite(v) for v in value), "Invalid paired fit score")
                    if phase == "heldout":
                        samples[task, name].append(sum(value) / n)
        check(len(streams) == 1, "Training streams differ across architectures within a seed")
    check(parameters["untied"] > parameters["tied"] and cfg["narrow_width"] < cfg["width"],
          "Unique parameter counts or control width do not match the paired design")
    contrasts = {}
    for task, metric in metrics.items():
        check(set(frames["pilot", task]).isdisjoint(frames["heldout", task]), "Pilot family reappeared in test")
        res = summary["contrasts"][task]["tied_minus_untied"][metric]
        differences = [a-b for a,b in zip(samples[task, "tied"], samples[task, "untied"])]
        check(len(res["by_seed"]) == len(seeds) and
              all(math.isclose(a,b,abs_tol=1e-6) for a,b in zip(differences,res["by_seed"])),
              "Paired seed differences disagree with held-out measurements")
        contrasts[task] = res
    return {"n_seeds": len(seeds), "parameters": parameters, "measurements": contrasts}


def audit(results, source):
    """Keep an independent status for each model/task/phase, including failures."""
    checks = [(engine, task, phase) for phase in PHASES for engine in ENGINES for task in TASKS]
    checks.append(("paired", "training", "control"))
    records = {}
    for engine, task, phase in checks:
        name = f"{engine}-{task}-{phase}"
        path = results / f"{name}.receipt.json"
        record = records[name] = {"status": "missing"}
        if not path.exists():
            failures = list(results.glob(f"{name}.*error*.json")) + list(results.glob(f"{name}.*fail*.json"))
            if failures:
                record.update(status="failed", files=sorted(p.name for p in failures))
            continue
        try:
            receipt = read_json(path)
            hardware = check_receipt(receipt, source)
            check((receipt["engine"], receipt["experiment"], receipt["phase"]) == (engine, task, phase),
                  "Receipt identifiers and filename disagree")
            if task == "training":
                measures = verify_paired(results, source)
            else:
                check_native(results, engine, receipt)
                checker = {"carry": verify_carry, "memory": verify_memory, "cue": verify_cue}[task]
                measures = checker(results, engine, phase)
            record.update(status="verified", receipt_sha256=file_hash(path),
                          hardware=hardware, **measures)
        except (OSError, ValueError, KeyError, TypeError, IndexError) as exc:
            record.update(status="invalid", reason=str(exc))
    for task in TASKS:
        for model in ENGINES:
            calibration, test = (records[f"{model}-{task}-{phase}"] for phase in PHASES)
            if "_families" in calibration and "_families" in test:
                if not calibration["_families"].isdisjoint(test["_families"]):
                    test.update(status="invalid", reason="Held-out family found in pilot")
        for phase in PHASES:
            pair = [records[f"{model}-{task}-{phase}"] for model in ENGINES]
            if all("family_data_sha256" in record for record in pair):
                if pair[0]["family_data_sha256"] != pair[1]["family_data_sha256"]:
                    for record in pair:
                        record.update(status="invalid", reason="Compared checkpoints scored different scene families")
    for record in records.values():
        record.pop("_families", None)
    return records


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("results", type=Path, help="Directory containing downloaded receipts and raw results")
    parser.add_argument("--source", type=Path, default=Path(__file__).resolve().parent,
                        help="Source snapshot to match against Kaggle receipts")
    parser.add_argument("--partial", action="store_true",
                        help="Allow absent jobs, but require at least one verified job and no invalid receipts")
    parser.add_argument("--json", type=Path, help="Write the full audit as JSON")
    args = parser.parse_args()
    if not args.results.is_dir():
        parser.error(f"Result directory does not exist: {args.results}")
    records = audit(args.results, args.source)
    counts = Counter(item["status"] for item in records.values())
    for name, record in records.items():
        detail = record.get("reason") or ", ".join(record.get("files", []))
        if record["status"] == "verified":
            detail = json.dumps(record.get("measurements"), ensure_ascii=False, sort_keys=True)
        print(f"{name:<24} {record['status']:<9} {detail}".rstrip())
    print(json.dumps(counts, sort_keys=True))
    if args.json:
        args.json.write_text(json.dumps({"jobs": records, "counts": counts}, indent=2,
                                        allow_nan=False) + "\n", encoding="utf-8")
    return int(bool(counts["invalid"] or counts["failed"] or not counts["verified"]
                    or (not args.partial and counts["missing"])))


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
waves = [[['ouro', '--phases', 'pilot'], ['qwen', '--phases', 'pilot'], ['paired']], [['ouro', '--phases', 'heldout', '--tasks', 'carry'], ['ouro', '--phases', 'heldout', '--tasks', 'memory'], ['ouro', '--phases', 'heldout', '--tasks', 'cue'], ['qwen', '--phases', 'heldout', '--tasks', 'carry'], ['qwen', '--phases', 'heldout', '--tasks', 'memory'], ['qwen', '--phases', 'heldout', '--tasks', 'cue']]]
import concurrent.futures
import hashlib
import json
import zipfile
import time
import queue

def archive_results(filename):
    path = work / filename
    tmp = path.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
        for p in sorted(out.rglob("*")):
            if p.is_file() and p.suffix != ".tmp":
                z.write(p, p.relative_to(work))
        for p in sorted(src.glob("*.py")):
            z.write(p, "source/" + p.name)
    tmp.replace(path)
    return path

# A shorter workload may use the next available card while the other continues.
slots = queue.Queue()
for card in range(torch.cuda.device_count()):
    slots.put(card)
def worker(args):
    card = slots.get()
    name = "-".join(arg for arg in args if not arg.startswith("--"))
    try:
        if "heldout" in args and not (out / f"{args[0]}-{args[-1]}-pilot.receipt.json").is_file():
            print(f"Skipped {name}: pilot failed", flush=True)
            return name, 1
        filename = out / f"{name}.log"
        with filename.open("w", buffering=1) as log:
            p = subprocess.Popen([sys.executable, "-u", str(src / "runner.py"),
                                  *args, "--out", str(out)], env=dict(env, CUDA_VISIBLE_DEVICES=str(card)),
                                 cwd=src, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            reported = 0.
            for line in p.stdout:
                log.write(line)
                event = any(k in line for k in ["BEGIN ", "COMPLETE ", "Native parity", "Error", "Exception", "Traceback"])
                if event or time.monotonic() - reported > 90:
                    print(f"[{name}] {line.rstrip()[:650]}", flush=True)
                    reported = time.monotonic()
            code = p.wait()
        print(f"{name} exit {code}", flush=True)
        return name, code
    finally:
        slots.put(card)
with concurrent.futures.ThreadPoolExecutor(max_workers=torch.cuda.device_count()) as pool:
    outcomes = {}
    for jobs in waves:
        pending = {pool.submit(worker, job) for job in jobs}
        while pending:
            done, pending = concurrent.futures.wait(pending, timeout=300,
                return_when=concurrent.futures.FIRST_COMPLETED)
            outcomes.update(f.result() for f in done)
            if pending or len(waves) > 1:
                archive_results("loop-progress.zip")
print(outcomes)
archive = archive_results("loop-results.zip")
print("Result archive", archive.stat().st_size, hashlib.sha256(archive.read_bytes()).hexdigest())
assert not any(outcomes.values()), outcomes

In [ ]:
import json
verification = subprocess.run([sys.executable, str(src / "verify_results.py"), str(out),
                               "--source", str(src), "--json", str(out / "audit.json")],
                              cwd=src, capture_output=True, text=True, env=env)
print(verification.stdout[-12000:], verification.stderr[-4000:], flush=True)
final_archive = archive_results("loop-results.zip")
print("Verified output bundle", final_archive.stat().st_size,
      hashlib.sha256(final_archive.read_bytes()).hexdigest(), flush=True)
verification.check_returncode()